# Reproducing Li, Daie, Svoboda and Druckmann (2016) from dandiset 000011

Paper. Li N, Daie K, Svoboda K, Druckmann S (2016). Robust neuronal dynamics in premotor cortex
during motor planning. *Nature* 532, 459-464.
[doi:10.1038/nature17643](https://doi.org/10.1038/nature17643). The claims file
(`claims/000011/10.1038-nature17643.yaml`) was written from the HHMI author manuscript in PubMed
Central (PMC5081260): main text, Methods and the legends of Figures 1 to 6 and Extended Data
Figures 1 to 10. The Corrigendum (doi:10.1038/nature18623) was not read.

Dandiset. [DANDI:000011](https://dandiarchive.org/dandiset/000011/0.220126.1907), version
0.220126.1907. The paper cites no DANDI version (its data statement points to CRCNS,
doi:10.6080/K0RB72JW), so the most recent published version is pinned. It is the only published
version. The draft lists the same 92 assets with the same total size (32,435,325,542 bytes); its
metadata was modified later (2026-02-12), which the analysis does not use.

What the dandiset holds (shown in the data check below). 91 extracellular files of 19 mice and
one empty behaviour file. 58 sessions of 12 mice are the control-mouse recordings of the paper
(experiment 1, unilateral photoinhibition, 20 sessions of 4 mice; experiments 2 and 3, bilateral
photoinhibition, 38 sessions of 7 and 3 mice). 33 sessions of 7 other mice have no photostimulus
record; their subjects and delay match the callosotomy experiment. The behavioural photoinhibition
experiments without recordings, the photoactivation mice and the untrained mice used to map
photoinhibition are not in the dandiset.

| claim | tested or blocked |
|---|---|
| unit-classification | tested |
| selective-fraction | tested |
| fig1d-late-delay-ipsilateral-bias | blocked: data_missing |
| fig1d-earlier-photoinhibition-minimal | blocked: data_missing |
| ed3-behaviour-recovery-interval | blocked: data_missing |
| ed4-return-to-trajectory | tested |
| fig1f-spike-rate-recovery | tested |
| fig1g-selectivity-recovery | tested |
| ed5-photoactivation-recovery | blocked: data_missing |
| ed5-photoactivation-rate-changes | blocked: data_missing |
| ed2b-silencing-strength | tested |
| ed2d-spatial-extent | blocked: data_missing |
| ed2e-no-rebound | tested |
| fig2b-unilateral-performance | blocked: data_missing |
| fig2b-bilateral-performance | blocked: data_missing |
| ed6a-bilateral-inconsistent-bias | tested |
| fig2-recordings-recovery | tested |
| fig2e-selectivity-recovery-times | tested |
| ed7-larger-photoinhibition | tested |
| ed7e-roc-larger-photoinhibition | tested |
| cd-stable-over-epochs | tested |
| cd-captures-selectivity | tested |
| fig3c-cd-recovery-roc | tested |
| ed8-roc-during-photoinhibition | tested |
| fig3c-cd-predicts-movement | tested |
| fig3d-persistent-mode | tested |
| fig3e-ramping-mode | tested |
| fig4-bilateral-cd-predicts-movement | tested |
| ed6b-session-bias-predicted | tested |
| ed9f-reaction-time | blocked: data_missing (analysis run) |
| ed9c-cd-window-robust | tested |
| fig5b-callosotomy-control-performance | blocked: data_missing |
| fig5b-callosotomy-unilateral-sensitivity | blocked: metadata_insufficient |
| ed10-callosotomy-preparatory-activity | blocked: metadata_insufficient |
| fig5-callosotomy-no-recovery | blocked: metadata_insufficient |
| fig5f-callosotomy-cd-reduced | blocked: metadata_insufficient |
| ed1-monolithic-models-fail | tested (simulation) |
| fig6-modular-models-recover | tested (simulation) |

Readings. Where the claims file lists readings of a detail, each is run, and a claim is
reproduced when it holds under one of them, as its criterion says. Ten questions were asked about
details the claims file did not give (`questions/000011.yaml`). A reader answered them on
2026-10-21: the paper gives none of them, and the claims file now names readings for each. For most,
the readings are method details whose outcomes must agree, else the claim is blocked as
`methods_underspecified`: lick-early trials dropped (K1) or kept (K2) in every neural analysis; the
spike counts that set a neuron's preferred trial type (D1 the whole trial, D2 the delay epoch, D3
each bin), the time that labels a bin (L1 its centre, L2 its end) and how far the search for
recovery runs (E1 to the go cue, E2 to 1.6 s after it); the selection test and bin of ed4 (s1, s2,
t1, t2); the Fisher table of ed6a (f1, f2); the trials whose average cd-captures-selectivity
decomposes (v1, v2); the sign of the ramping mode (g1, g2); the noise and step of the modular
attractor (n1, n2, h1, h2); and the lick threshold of ed9f. Two answers fix a reading: trials are
counted per condition for convention D (N1) and for convention A (T1), and counting per condition
and trial type (N2, T2) is shown as a variant. Each section says what its answer changed.

First verification round. A verifier reviewed the attempt on 2026-10-21. In reply, the notebook
adds the outputs it asked for (upsampled spike widths, the size of the lick-early artefact under
K2, per-neuron trial counts and epoch selectivity of the included populations, per-session ROC of
modes 2 and 4, raw lick-trace summaries and example traces, raw levels above the experiment-2
baseline, experiment-3 behaviour and the ed9f test on experiment 3, and the ed10 counts on all
trials), and three outcomes changed: selective-fraction and ed2b-silencing-strength are not
reproduced (partial before), and cd-stable-over-epochs is blocked as `methods_underspecified`
(reproduced before), as the trials that give w are not fixed. Three new questions were asked
(`questions/000011.yaml`).

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import itertools
import time
import warnings

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import remfile
from scipy import stats

import li2016 as L

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
# LOAD: stream every file of the pinned version (trials, units, photostimulus record)
started = time.time()
assets, results = L.load_all()
S, U = L.assemble(results)
print(f"loaded {len(results)} extracellular files of dandiset {L.DANDISET} version {L.VERSION} in {time.time() - started:.0f} s")

## Data check

In [ ]:
print("Assets in the version:", len(assets))
print(assets.assign(kind=np.where(assets.path.str.contains("ecephys"), "behavior+ecephys+ogen", "behavior only"))
      .groupby("kind").size().to_string())
# The one behaviour-only file: list everything it holds besides the format specification
beh = assets[~assets.path.str.contains("ecephys")].iloc[0]
with h5py.File(remfile.File(L.s3_url(beh.asset_id)), "r") as f:
    items = []
    f.visititems(lambda n, o: items.append((n, getattr(o, "shape", None))) if not n.startswith("specifications") else None)
print(f"\n{beh.path}: {len(items)} objects outside /specifications")
for n, sh in items:
    print("  ", n, sh if sh is not None else "(group)")

In [ ]:
ses = pd.DataFrame([{"sid": s["sid"], "subject": s["subject"], "genotype": "VGAT" if "VGAT" in s["genotype"] else "PV x ReaChR",
                     "experiment": s["exp"], "trials": len(s["trials"]),
                     "behaving_trials": int(s["trials"].behaving.sum()),
                     "units": int((U.sid == s["sid"]).sum()), "sites": ",".join(s["meta"]["sites"]),
                     "trials_overlapping": s["meta"]["trials_overlapping"]} for s in S.values()])
by_subject = ses.groupby(["subject", "genotype"]).agg(sessions=("sid", "size"), experiments=("experiment", lambda x: ",".join(sorted(set(x)))),
                                                     units=("units", "sum")).reset_index()
print(by_subject.to_string(index=False))
grp = ses.assign(group=np.where(ses.experiment == "no_photostim_record", "no photostimulus record", "control mice"))
print("\nSessions, mice and units by group:")
print(grp.groupby("group").agg(sessions=("sid", "size"), mice=("subject", "nunique"), units=("units", "sum")).to_string())
print("\nSessions by experiment:")
print(ses.groupby("experiment").agg(sessions=("sid", "size"), mice=("subject", "nunique"), units=("units", "sum")).to_string())
print("\nGenotypes of the control mice:", grp[grp.group == "control mice"].groupby("genotype").subject.nunique().to_dict())
print("Genotypes of the mice without a photostimulus record:", grp[grp.group != "control mice"].groupby("genotype").subject.nunique().to_dict())
print("\nPaper (Methods): 58 sessions (20 unilateral, 38 bilateral) of 12 mice (5 VGAT, 7 PV x ReaChR);"
      " callosotomy 7 mice (6 VGAT, 1 PV x ReaChR); Extended Data Fig. 2b: experiment 1 from 6 mice,"
      " experiment 2 from 7, experiment 3 from 3.")

The subjects split in two groups. 12 mice (5 VGAT-ChR2-EYFP, 7 PV x ReaChR) with 58 sessions are
the control mice of the paper: their counts match the Methods. The 33 sessions of the 7 other
mice (6 VGAT, 1 PV x ReaChR) have no photostimulus record at all; the genotype counts and the
1.7 s delay match the callosotomy experiment, but no file says these mice were callosotomised
(the subject description holds only the JAX strains). Experiment 1 holds 4 mice here, where
Extended Data Fig. 2b gives 6. Three experiment-1 sessions of mouse 255200 have no photostimulus
record either. The behaviour-only file holds only metadata.

In [ ]:
rows = []
for s in S.values():
    tr = s["trials"]
    b = tr[tr.behaving]
    ne = b[~b.early_lick]
    rows.append({"experiment": s["exp"], "sid": s["sid"], "behaving": len(b), "early_lick": int(b.early_lick.sum()),
                 "sample_s_median": np.median(ne.delay - ne["sample"]) if len(ne) else np.nan,
                 "delay_s_median": np.median(ne.go - ne.delay) if len(ne) else np.nan,
                 "delay_off_nominal_noearly": int((abs((ne.go - ne.delay) - np.median(ne.go - ne.delay)) > 0.05).sum()) if len(ne) else 0,
                 "delay_off_nominal_early": int((abs((b[b.early_lick].go - b[b.early_lick].delay) - np.median(ne.go - ne.delay)) > 0.05).sum()) if len(ne) else 0,
                 **{o: int((b.outcome == o).sum()) for o in ["hit", "miss", "ignore"]},
                 "non_performing": int((tr.outcome == "non-performing").sum()),
                 "trials_overlapping": s["meta"]["trials_overlapping"]})
tt = pd.DataFrame(rows)
print("Behaving trials (sample, delay and go events and an outcome); session 258250_20141202 is set to none (below).")
print(tt.groupby("experiment").agg(sessions=("sid", "size"), behaving=("behaving", "sum"), early_lick=("early_lick", "sum"),
                                   hit=("hit", "sum"), miss=("miss", "sum"), ignore=("ignore", "sum"), non_performing=("non_performing", "sum"),
                                   sample_s=("sample_s_median", lambda x: f"{x.min():.2f}-{x.max():.2f}"),
                                   delay_s=("delay_s_median", lambda x: f"{x.min():.2f}-{x.max():.2f}"),
                                   off_nominal_noearly=("delay_off_nominal_noearly", "sum"),
                                   off_nominal_early=("delay_off_nominal_early", "sum")).to_string())
print("\nSessions whose trials overlap in time:")
print(tt[tt.trials_overlapping > 0][["sid", "trials_overlapping"]].to_string(index=False))
bad = S["258250_20141202"]["trials"]
print("\n258250_20141202, trial ids in time order (first 30 behaving or not):", bad.trial_id.values[:30].tolist())
print("task and protocol values over all files:", sorted(set(np.concatenate([s["trials"].task.unique() for s in S.values()]))),
      sorted(set(np.concatenate([s["trials"].protocol.unique() for s in S.values()]))))

In every session the sample epoch (pole onset to delay onset) lasts 1.3 s and the delay 1.3 s
(experiment 1) or 1.7 s (experiments 2, 3 and the sessions without a photostimulus record), as the
Methods say, in trials without an early lick. In early-lick trials the delay restarts, so their
delay differs from the nominal one. Session 258250_20141202 holds two interleaved sequences of
trial ids whose times overlap (98 overlapping pairs), so its trial events cannot be assigned to
trials; it is left out of every trial-based analysis and its units are counted only. Two sessions of
mouse 255200 have one or two overlapping pairs and are kept. The trial
table has no column for the photostimulus: conditions are read from the laser record below.

In [ ]:
ps = []
for s in S.values():
    m = s["meta"]
    if not m["sites"]:
        continue
    tr = s["trials"]
    ps.append({"sid": s["sid"], "experiment": s["exp"], "events": m["ps_events"], "events_without_time": m["ps_events_nan_time"],
               "laser_chunks": sum(m[f"chunks_{x}"] for x in m["sites"]),
               "chunks_outside_trials": sum(m[f"chunks_outside_{x}"] for x in m["sites"]),
               "chunk_trials_without_event": int((tr.ps_site.notna() & tr.ps_on.isna()).sum()),
               "of_them_behaving": int((tr.ps_site.notna() & tr.ps_on.isna() & tr.behaving).sum()),
               "event_trials_without_chunk": int((tr.ps_site.isna() & tr.ps_on.notna()).sum()),
               "samples_per_chunk": {x: m[f"samples_{x}"] / m[f"chunks_{x}"] for x in m["sites"]}})
ps = pd.DataFrame(ps)
print(ps.drop(columns="samples_per_chunk").groupby("experiment").sum(numeric_only=True).to_string())
print("Samples per laser chunk (all sites, all sessions):", sorted(set(v for d in ps.samples_per_chunk for v in d.values())))
print("Sessions with event trials lacking a laser chunk:", ps[ps.event_trials_without_chunk > 0][["sid", "event_trials_without_chunk"]].values.tolist())
shape = []
for s in S.values():
    for k, v in s["meta"].items():
        if k.startswith("shape_"):
            shape.append({"experiment": s["exp"], "site": k[6:], "mean_mW": round(v[0], 2), "cv": round(v[1], 2)})
shape = pd.DataFrame(shape)
print("\nLaser power 0.1-0.5 s after onset (mW) in the first trial of each site and session:")
print(shape.groupby(["experiment", "site"]).agg(sessions=("cv", "size"), mean_mW=("mean_mW", lambda x: f"{x.min()}-{x.max()}"),
                                                cv=("cv", lambda x: f"{x.min()}-{x.max()}")).to_string())
allt = pd.concat([s["trials"].assign(experiment=s["exp"], sid=s["sid"]) for s in S.values() if s["meta"]["sites"]], ignore_index=True)
light = allt[allt.ps_on.notna()]
print("\nPeak power of photostimulus events (mW), counts:", np.histogram(light.ps_power, bins=[0, 1, 2.5, 3.5, 5, 7, 11, 14])[0].tolist(),
      "for bins [0, 1, 2.5, 3.5, 5, 7, 11, 14]")

The photostimulus of a trial is recorded twice: as `photostim_start_time` and
`photostim_stop_time` events (the event value is the peak power, mW) and as 5 s chunks of the laser
power and AOM traces (10 kHz, 50,000 samples per chunk), one chunk per photostimulus trial, in one
series per site (left, right, bilateral). The chunks give the site; the events give the time. The
two agree trial by trial: every event with a time has a chunk except in two sessions, and the
chunks without an event fall in non-performing trials. Event values below 1 mW (77 events) are
kept apart as `no_light`. The traces separate the protocols of the Methods: a 40 Hz sinusoid of
1.5 mW average for one spot, and a constant power of 1.5 mW per spot when the beam is scanned
(3.1 mW for one spot per side, 6.1 mW for four spots on one side, 12.6 mW for four spots per
side). Experiment 2 sessions use one spot left or right and eight spots bilaterally; experiment 3
sessions use two spots bilaterally (one per side) and four spots left or right, or one spot left
or right in 6 of its 12 sessions.

In [ ]:
lab = allt[allt.behaving & ~allt.early_lick]
print("Behaving trials without an early lick, by experiment and condition:")
print(pd.crosstab(lab.cond, lab.experiment).to_string())
lp = lab[lab.ps_on.notna() & ~lab.cond.isin(["no_light", "ps_unknown"])]
print("\nLight onset and offset re go cue (s), and duration, by condition (trials without an early lick):")
print(lp.groupby(["experiment", "cond"]).agg(n=("ps_on", "size"), onset=("ps_on_re_go", lambda x: f"{x.min():.3f} to {x.max():.3f}"),
                                             offset=("ps_off_re_go", lambda x: f"{x.min():.3f} to {x.max():.3f}"),
                                             duration=("ps_off", lambda x: "")).drop(columns="duration").to_string())
print("Light duration (s), all conditions:", np.unique(np.round(lp.ps_off - lp.ps_on, 3)).tolist())
print("Onset re delay onset (s) in experiments 2 and 3:", np.unique(np.round((lp.ps_on - lp.delay)[lp.experiment != 'exp1'], 3)).tolist())
print("Fraction of behaving trials with light, per session (control sessions):",
      allt[allt.behaving].groupby("sid").apply(lambda d: (~d.cond.isin(["control"])).mean()).describe().round(2).to_dict())

Experiment 1 holds three conditions, 0.5 s of light ending 1.6 s (sample), 0.8 s (early delay) and
0.3 s (late delay) before the go cue. This settles the naming question of the claims file
(convention T): the recorded "late delay" condition ends 0.3 s before the go cue (the Methods'
naming); the condition ending at the go cue (the Results' "late delay" of Fig. 1d) is not in the
recordings. Experiments 2 and 3 have 0.8 s of light from delay onset. The stop event is 0.5 s or
0.8 s after the start event, so it marks the end of the ramp, the offset the claims file uses.

In [ ]:
# Event time against the light in the trace: two sessions read in full
for sid in ["255200_20140913", "263808_20150121"]:
    s = S[sid]; tr = s["trials"]
    aid = assets[assets.path == s["path"]].asset_id.iloc[0]
    rows = []
    with h5py.File(remfile.File(L.s3_url(aid)), "r") as f:
        for site in s["meta"]["sites"]:
            g = f[f"/stimulus/presentation/{site} ALM_laser_power"]
            d, ts = g["data"][:], g["timestamps"][:]
            for r in tr[(tr.ps_site == site) & tr.ps_on.notna() & tr.behaving].itertuples():
                c = int(r.ps_chunk); seg = d[c * 50000:(c + 1) * 50000]; t = ts[c * 50000:(c + 1) * 50000]
                on = np.where(seg > 0.5)[0]
                rows.append({"early_lick": r.early_lick, "event_minus_light_ms": (r.ps_on - t[on[0]]) * 1000 if len(on) else np.nan,
                             "delay_event_minus_light_ms": (r.delay - t[on[0]]) * 1000 if len(on) else np.nan})
    df = pd.DataFrame(rows)
    df["mismatch"] = df.event_minus_light_ms.abs() > 20
    print(sid, "trials:", len(df))
    print(df.groupby("early_lick").agg(n=("mismatch", "size"), mismatch=("mismatch", "sum"),
                                       event_minus_light_ms=("event_minus_light_ms", lambda x: f"{x.min():.0f} to {x.max():.0f}")).to_string())

In trials without an early lick the photostimulus event and the light in the trace agree to
within 5 ms. In early-lick trials of these two sessions the event is logged up to about 1.6 s (1587 ms) after the light, at the
restarted delay, so in those trials the event times do not mark the light. This is one reason
the readings below that keep early-lick trials are kept apart.

In [ ]:
# Size of the lick-early artefact (verifier's request). Under K2 lick-early trials enter with their
# events at their logged times (claims file, convention K). Per condition: how many lick-early trials
# K2 adds (behaving, correct or incorrect), and where the light seen in the laser trace falls against
# the logged go cue and the logged photostimulus event. Every lick-early photostimulus trial of the
# labelled sessions is read (one 5 s trace chunk per trial).
t0 = time.time()
k2rows = []
for sid, s in S.items():
    if s["exp"] not in ("exp1", "exp2", "exp3"):
        continue
    tr = s["trials"]
    keep = L.trial_mask(tr, "keep")
    e = tr[keep & tr.early_lick.values & tr.ps_site.notna().values & tr.ps_on.notna().values & ~tr.cond.isin(["no_light", "ps_unknown", "control"]).values]
    for c, n in pd.Series(tr.cond[keep]).value_counts().items():
        k2rows.append({"experiment": s["exp"], "sid": sid, "cond": c, "kind": "count", "trials_K2": int(n),
                       "early_K2": int((keep & tr.early_lick.values & (tr.cond == c).values).sum())})
    if len(e):
        aid = assets[assets.path == s["path"]].asset_id.iloc[0]
        lt = L.light_onsets(aid, e)
        for r, x in zip(e.itertuples(), lt):
            k2rows.append({"experiment": s["exp"], "sid": sid, "cond": r.cond, "kind": "light",
                           "light_re_go": x - r.go, "event_re_go": r.ps_on - r.go, "event_minus_light": r.ps_on - x})
k2t = pd.DataFrame(k2rows)
cnt = k2t[k2t.kind == "count"].groupby(["experiment", "cond"])[["trials_K2", "early_K2"]].sum()
cnt["early_fraction"] = (cnt.early_K2 / cnt.trials_K2).round(3)
lt = k2t[k2t.kind == "light"]
q = lt.groupby(["experiment", "cond"]).agg(
    read=("light_re_go", "size"), light_found=("light_re_go", lambda x: int(x.notna().sum())),
    light_re_go=("light_re_go", lambda x: f"{x.min():.2f} / {x.median():.2f} / {x.max():.2f}"),
    event_re_go=("event_re_go", lambda x: f"{x.min():.2f} / {x.median():.2f} / {x.max():.2f}"),
    event_after_light_over_0_1s=("event_minus_light", lambda x: int((x > 0.1).sum())),
    event_minus_light_median_s=("event_minus_light", "median"))
q = q.join(cnt.trials_K2)
q["displaced_fraction_of_condition"] = q.event_after_light_over_0_1s / q.trials_K2
print(f"read in {time.time() - t0:.0f} s")
print("Trials per condition under K2 (behaving, correct or incorrect) and the lick-early trials among them:")
print(cnt.to_string())
print("\nLick-early photostimulus trials: light onset and logged event re the logged go cue (s; min / median / max),")
print("and how many have the event more than 0.1 s after the light:")
print(q.round(3).to_string())
print("Lick-early photostimulus trials read:", len(lt), "; with the event more than 0.1 s after the light:", int((lt.event_minus_light > 0.1).sum()),
      "; earliest light re the logged go cue (s):", round(lt.light_re_go.min(), 2))

Under K2 the lick-early trials are 15% to 22% of the trials of each condition in experiments 1
and 2, and 7% to 14% in experiment 3. In most of them the light falls where the logged events put
it, but in some it came earlier, before one or more restarts of the delay (light as early as 10 s
before the logged go cue). Those trials, in which the logged photostimulus event lies more than
0.1 s after the light, are 16% of the sample-epoch condition of experiment 1, 3% to 4% of its other
two conditions, 6.5% to 7.5% of each photoinhibition condition of experiment 2 and 1% to 4% in
experiment 3 (`displaced_fraction_of_condition`). In those trials the windows aligned to the
logged events miss the light, so K2 mixes trials without photoinhibition into the photoinhibition
averages. The K2 readings are run as the claims file defines them (events at their logged times);
the claims whose outcome depends on K2 say so. A question asks the reader whether K2 should align
those trials to the light in the trace.

In [ ]:
ctl_units = U[U.exp != "no_photostim_record"]
print("Units in control mice:", len(ctl_units), "; without a photostimulus record:", int((U.exp == "no_photostim_record").sum()))
print("\nWidth class (trough to peak of waveform_mean, Methods thresholds) against the files' cell_type:")
print(pd.crosstab(U.cls, U.cell_type).to_string())
print("\nTrough-to-peak widths (ms) and counts, control mice:")
w = ctl_units.width_ms.round(3).value_counts().sort_index()
print(w.to_string())
print("\nWaveform samples:", sorted(set(len(x) for x in U.waveform)), "; sampling rate (Hz):", sorted(set(U.fs)),
      "; Methods: 19531.25 Hz")
print("Waveform_mean range (stated unit not given in the file):", f"{np.min([x.min() for x in U.waveform]):.2e} to {np.max([x.max() for x in U.waveform]):.2e}")
print("Unit quality values:", U.quality.value_counts().to_dict())
print("Units whose spike times are not in time order in the file:", {s["sid"]: s["meta"]["units_unsorted"] for s in S.values() if s["meta"]["units_unsorted"]},
      "; sessions whose trials are not in time order:", {s["sid"]: s["meta"]["trials_unsorted"] for s in S.values() if s["meta"]["trials_unsorted"]})
outside = []
for u in U.itertuples():
    obs = u.obs
    inside = np.zeros(len(u.spikes), bool)
    for a, b in obs:
        inside |= (u.spikes >= a) & (u.spikes <= b)
    outside.append((~inside).mean())
print("Fraction of spikes outside the unit's observation intervals: max", f"{max(outside):.4f}")
obs_frac = [o[S[s]["trials"].behaving.values].mean() for o, s in zip(U.observed, U.sid) if S[s]["trials"].behaving.any()]
print("Fraction of behaving trials in which a unit is observed: median", f"{np.median(obs_frac):.2f}", "min", f"{np.min(obs_frac):.2f}")
print("Mean firing rate over observation intervals (Hz), control mice:",
      pd.Series([len(sp) / max(np.sum(o[:, 1] - o[:, 0]), 1e-9) for sp, o in zip(ctl_units.spikes, ctl_units.obs)]).describe().round(2).to_dict())

The files' `cell_type` labels follow the same thresholds as the Methods (every unit wider than
0.45 ms is `Pyr`), except that the units of intermediate width are labelled `FS` (45 in all; the 15
in the control mice are all 0.358 ms wide). At 19531.25 Hz the width takes discrete values 0.051 ms apart; none falls on a threshold.
In 28 units of 3 sessions the spike times are not in time order (the files store them in trial
order, and two of these sessions have trials out of time order); they are sorted here. A few spikes
(at most 3.3% of a unit's) fall outside the unit's observation intervals.

In [ ]:
# Clock check 1: spikes against the photostimulus record. Population rate in the light window
# (onset to offset minus ramp) over the control rate in the same window, and the same with the
# window shifted 1.5 s later (after the light).
rows = []
for sid, s in S.items():
    if s["exp"] not in ("exp1", "exp2", "exp3") or s["meta"]["trials_overlapping"]:
        continue
    tr = s["trials"]; m = L.trial_mask(tr, "drop")
    uu = U[(U.sid == sid) & (U.cls == "pyr")]
    ipsi = {"exp1": ["sample", "early_delay", "late_delay"], "exp2": ["left1"], "exp3": ["left4", "bilateral2"]}[s["exp"]]
    for c in ipsi:
        p = m & (tr.cond == c).values; k = m & (tr.cond == "control").values
        if p.sum() < 3:
            continue
        on, off = np.median(tr.ps_on_re_go[p]), np.median(tr.ps_off_re_go[p]) - (0.1 if s["exp"] == "exp1" else 0.2)
        for shift in (0.0, 1.5):
            a = sum(L.counts_in(u.spikes, tr.go.values[p], on + shift, off + shift).mean() for u in uu.itertuples())
            b = sum(L.counts_in(u.spikes, tr.go.values[k], on + shift, off + shift).mean() for u in uu.itertuples())
            rows.append({"sid": sid, "experiment": s["exp"], "cond": c, "shift_s": shift, "ratio": a / b if b else np.nan})
clk = pd.DataFrame(rows)
print("Population rate, photoinhibition over control, in the light window and 1.5 s later (sessions):")
print(clk.groupby(["experiment", "cond", "shift_s"]).ratio.describe()[["count", "min", "50%", "max"]].round(2).to_string())

In [ ]:
# Clock check 2: trial type against spikes. Fraction of putative pyramidal units whose delay-epoch
# counts differ between lick left and lick right control trials (t-test, p < 0.05), with the true
# trial types and with the trial types of the next trial (a shift that breaks the link).
rows = []
for sid, s in S.items():
    if s["meta"]["trials_overlapping"] or s["exp"] == "no_photostim_record" and False:
        continue
    tr = s["trials"]; m = L.trial_mask(tr, "drop", ("hit",)) & tr.cond.isin(["control", "unlabelled"]).values
    if m.sum() < 20:
        continue
    t = tr[m]; right = (t.instruction == "right").values
    shifted = np.roll(right, 1)
    uu = U[(U.sid == sid) & (U.cls == "pyr")]
    for u in uu.itertuples():
        ok = u.observed[m]
        c = L.counts_in(u.spikes, t.go.values, (t.delay - t.go).values, np.zeros(len(t)))
        rows.append({"sid": sid, "experiment": s["exp"], "true": L.ttest(c[ok & right], c[ok & ~right]) < 0.05,
                     "shifted": L.ttest(c[ok & shifted], c[ok & ~shifted]) < 0.05})
dec = pd.DataFrame(rows)
print("Fraction of units selective in the delay epoch, true against shifted trial types:")
print(dec.groupby("experiment")[["true", "shifted"]].mean().round(3).assign(units=dec.groupby("experiment").size()).to_string())

In [ ]:
# Sessions without a photostimulus record: per-trial population rate in the window where the light
# falls in the other sessions (experiment 1: the three windows; otherwise delay onset + 0.0-0.6 s),
# over the session median. Trials below 0.3 are counted; the same count 1.5 s later is the reference.
rows = []
for sid, s in S.items():
    if s["exp"] not in ("no_photostim_record", "exp1_unlabelled"):
        continue
    tr = s["trials"]; m = L.trial_mask(tr, "drop", ("hit", "miss", "ignore"))
    t = tr[m]
    uu = U[(U.sid == sid)]
    if len(t) < 10 or len(uu) == 0:
        rows.append({"sid": sid, "experiment": s["exp"], "trials": len(t)}); continue
    if s["exp"] == "exp1_unlabelled":
        wins = [(-2.1, -1.7), (-1.3, -0.9), (-0.8, -0.4)]
    else:
        wins = [(-1.7, -1.1)]
    out = {"sid": sid, "experiment": s["exp"], "trials": len(t), "units": len(uu)}
    for shift, name in ((0.0, "silenced"), (1.5, "silenced_1.5s_later")):
        low = np.zeros(len(t), bool)
        for a, b in wins:
            pop = sum(L.counts_in(u.spikes, t.go.values, a + shift, b + shift) for u in uu.itertuples())
            med = np.median(pop)
            low |= pop < 0.3 * med if med > 0 else np.zeros(len(t), bool)
        out[name] = round(low.mean(), 2)
    rows.append(out)
unl = pd.DataFrame(rows)
print(unl.to_string(index=False))
print("\nMedian over sessions of the fraction of trials below 30% of the session median, in the light window and 1.5 s later:")
print(unl.groupby("experiment")[["silenced", "silenced_1.5s_later"]].median().to_string())
print("Sessions where the fraction in the light window exceeds the one 1.5 s later by 0.1 or more:",
      int(((unl.silenced - unl["silenced_1.5s_later"]) >= 0.1).sum()), "of", int(unl.silenced.notna().sum()))

The clock checks hold for the 54 sessions with a photostimulus record. In each ipsilateral
condition the population rate in the light window has a median over sessions of 0.03 to 0.05 of the
control rate (one experiment-2 session reaches 0.70, and one experiment-3 session 1.26 under
bilateral light), and 1.5 s later the median is 0.94 to 1.09, so the spikes, the trial events and the
photostimulus record share one clock. Delay-epoch selectivity is found in 0.30 to 0.39 of units with
the true trial types against 0.06 to 0.09 with those of the next trial, in every group except the
three unlabelled experiment-1 sessions (0.33 against 0.23, 30 units).

In most sessions without a photostimulus record, a fraction of trials (up to about 0.4) have their
population rate fall below 30% of the session median in the light window of the other sessions,
far more than 1.5 s later. These sessions therefore contain photoinhibition trials that no file
labels (in a few sessions no such trials appear in this window, so their light, if any, fell
elsewhere). Their
control trials cannot be told from their photoinhibition trials, so no analysis of them that needs
control trials or conditions can be run.

## unit-classification

> 1012 single units were recorded in left ALM during behaviour in 58 sessions (20 unilateral, 38
> bilateral experiment sessions); by spike width 72 were fast-spiking, 890 putative pyramidal, and
> 50 intermediate and excluded.

Criterion: width is the trough-to-peak interval of the mean waveform; units and sessions of the
control mice only. Reproduced when all values are within tolerance.

In [ ]:
ctl = U[U.exp.isin(["exp1", "exp1_unlabelled", "exp2", "exp3"])]
expected = {"single units": (1012, 101.2), "recording sessions": (58, 5.8), "fast-spiking units": (72, 7.2),
            "putative pyramidal units": (890, 89), "intermediate units": (50, 5)}
measured = {"single units": len(ctl), "recording sessions": ctl.sid.nunique(), "fast-spiking units": int((ctl.cls == "FS").sum()),
            "putative pyramidal units": int((ctl.cls == "pyr").sum()), "intermediate units": int((ctl.cls == "intermediate").sum())}
uc = pd.DataFrame([{"quantity": k, "paper": v[0], "tolerance": v[1], "measured": measured[k],
                    "within": abs(measured[k] - v[0]) <= v[1]} for k, v in expected.items()])
print(uc.to_string(index=False))
print("\nSessions: unilateral (experiment 1)", ctl[ctl.exp.str.startswith("exp1")].sid.nunique(),
      "; bilateral (experiments 2 and 3)", ctl[ctl.exp.isin(["exp2", "exp3"])].sid.nunique())
print("Units by mouse and class:")
print(pd.crosstab(ctl.subject, ctl.cls, margins=True).to_string())
cal = U[U.exp == "no_photostim_record"]
print("\nCounted apart, mice without a photostimulus record:", len(cal), "units,", cal.sid.nunique(), "sessions;",
      cal.cls.value_counts().to_dict())
print("Units missing against the paper's 1012:", 1012 - len(ctl), "; intermediate units missing against its 50:", 50 - measured["intermediate units"])
# Variant (verifier's request): widths from waveforms interpolated to 10 times the stored rate (cubic
# spline), to see whether the 19.5 kHz sampling explains the few intermediate units. The criterion's
# measure is the stored-rate width above.
ctl = ctl.assign(width_up_ms=[L.spike_width_upsampled_ms(w, fs) for w, fs in zip(ctl.waveform, ctl.fs)])
ctl["cls_up"] = np.where(ctl.width_up_ms < L.FS_MAX_MS, "FS", np.where(ctl.width_up_ms > L.PYR_MIN_MS, "pyr", "intermediate"))
print("\nVariant, upsampled widths: class counts", ctl.cls_up.value_counts().to_dict(), "(paper FS 72, pyramidal 890, intermediate 50)")
print("Stored-rate class against upsampled class:")
print(pd.crosstab(ctl.cls, ctl.cls_up).to_string())
print("Upsampled widths (ms) of control-mouse units, counts in 0.05 ms bins from 0.20 to 0.60:",
      np.histogram(ctl.width_up_ms, bins=np.round(np.arange(0.2, 0.6001, 0.05), 2))[0].tolist(),
      "; below 0.20:", int((ctl.width_up_ms < 0.2).sum()), "; above 0.60:", int((ctl.width_up_ms > 0.6).sum()))


**Result: partially reproduced.** The control mice hold 979 units in 58 sessions (20 unilateral, 38
bilateral), with 74 fast-spiking and 890 putative pyramidal units, all within tolerance; 890 is the
paper's number exactly. Only 15 units have intermediate widths, against 50: the dandiset holds 33
units fewer than the paper's 1012, about the number of intermediate units missing (35), so the
intermediate units were probably left out of the deposit.

With widths from waveforms interpolated to ten times the stored rate (a variant), 11 units are
intermediate, 79 fast-spiking and 889 putative pyramidal, so the 19.5 kHz sampling does not
explain the shortfall of intermediate units. The criterion defines no partial outcome; four of its
five values hold, so the outcome is partial by the rule given in the Summary.

## selective-fraction

> Most putative pyramidal neurons distinguished trial types (634 of 890); 176 in the sample epoch,
> 337 in the delay epoch and 493 in the response epoch.

Criterion: convention S on the putative pyramidal neurons, control trials; reproduced when all four
counts are within tolerance under one combination of readings: response epoch 1.3 s (R1) or 1.5 s
(R2) after the go cue; correct trials (O1) or correct and incorrect (O2); Student's or Welch's
t-test. Lick-early trials are dropped (K1) or kept (K2); the answer (convention K) found the paper
silent, so the outcome must hold under both. The 44 putative pyramidal units of the
three experiment-1 sessions without a photostimulus record have no identifiable control trials;
they are left out, and shown as a variant with all their trials.

In [ ]:
pyr_ctl = ctl[ctl.cls == "pyr"]
pyr_lab = pyr_ctl[pyr_ctl.exp.isin(["exp1", "exp2", "exp3"])]
SEL, sel_rows, tests = {}, [], {}
for early in ("drop", "keep"):
    SEL[early] = {}
    for r in L.S_READINGS:
        st = L.selectivity_tests(pyr_lab, S, r["response"], r["outcomes"], early, r["equal_var"])
        tests[(early, L.s_key(r))] = st
        sig = st[["sample", "delay", "response"]] < 0.05
        SEL[early][L.s_key(r)] = set(st.uid[sig.any(axis=1)])
        sel_rows.append({"early_lick": early, "reading": L.s_key(r), "tested": len(st),
                         "untestable": int(st[["sample", "delay", "response"]].isna().any(axis=1).sum()),
                         "any": int(sig.any(axis=1).sum()), "sample": int(sig["sample"].sum()),
                         "delay": int(sig.delay.sum()), "response": int(sig.response.sum())})
sf = pd.DataFrame(sel_rows)
exp_sf = {"any": (634, 63.4), "sample": (176, 17.6), "delay": (337, 33.7), "response": (493, 49.3)}
for k, (v, tol) in exp_sf.items():
    sf[k + "_ok"] = (sf[k] - v).abs() <= tol
sf["all_four"] = sf[[k + "_ok" for k in exp_sf]].all(axis=1)
print("Putative pyramidal units with labelled trials:", len(pyr_lab), "of", len(pyr_ctl))
print(sf.to_string(index=False))
print("\nCombinations with all four within tolerance:", int(sf.all_four.sum()), "of", len(sf))
sf["n_within"] = sf[[k + "_ok" for k in exp_sf]].sum(axis=1)
print("Most counts within tolerance in one reading, per lick-early reading:", sf.groupby("early_lick").n_within.max().to_dict(),
      "; readings with three of four:", sf[sf.n_within == 3][["early_lick", "reading"]].values.tolist())
print("Largest sample-epoch count:", sf["sample"].max(), "; tolerance band", 176 - 17.6, "to", 176 + 17.6)

In [ ]:
# Variant: the 44 units of the three unlabelled sessions added, with all their trials as control
rows = []
for early in ("drop", "keep"):
    for r in L.S_READINGS:
        st = L.selectivity_tests(pyr_ctl, S, r["response"], r["outcomes"], early, r["equal_var"], conds=("control", "unlabelled"))
        sig = st[["sample", "delay", "response"]] < 0.05
        row = {"early_lick": early, "reading": L.s_key(r), "any": int(sig.any(axis=1).sum()), "sample": int(sig["sample"].sum()),
               "delay": int(sig.delay.sum()), "response": int(sig.response.sum())}
        row["all_four"] = all(abs(row[k] - v) <= tol for k, (v, tol) in exp_sf.items())
        rows.append(row)
sfv = pd.DataFrame(rows)
print("Variant with the 44 units of the unlabelled sessions (all their trials):")
print(sfv.to_string(index=False))
print("By mouse, reading R1-O1-student, lick-early dropped (labelled sessions):")
st = tests[("drop", "R1-O1-student")].merge(pyr_lab[["uid", "subject"]], on="uid")
print(st.assign(sel=(st[["sample", "delay", "response"]] < 0.05).any(axis=1)).groupby("subject").sel.agg(["sum", "size"]).T.to_string())

**Result: not reproduced.** On the 846 putative pyramidal units whose control trials can be
identified, no combination of readings brings all four counts within tolerance. With lick-early
trials dropped (K1) only the response-epoch count reaches its band (445 or 446 under three readings
with correct trials only); the counts in any epoch (at most 565, band 571 to 697), in the sample
epoch (at most 156, band 158 to 194) and in the delay epoch (at most 294, band 303 to 371) fall
short. With lick-early trials kept (K2) three of four reach their bands under the four readings
with correct trials only (any epoch 575 to 588, delay 315 to 320, response 453 to 464), and the
sample count does not (at most 149). The outcome must be the one K1 and K2 both give, and under K1
only one of the four counts holds, so the claim is not reproduced (the first round gave partial;
the rule is in the Summary). Most units are selective all the same (563 of 846 with K1, R1 and
correct trials). With the 44 units of the three unlabelled sessions added on all their trials, which
include photoinhibition trials and so are not the control trials the criterion names, all four
counts are within tolerance with correct trials and K1 (592, 165, 307 and 464 with R1 and Student's
test) and under no reading with K2.

## Behaviour during the recordings (variant for blocked claims)

fig1d-late-delay-ipsilateral-bias, fig1d-earlier-photoinhibition-minimal,
ed3-behaviour-recovery-interval and the two fig2b claims rest on the paper's behavioural
photoinhibition experiments (5 and 13 mice, photoinhibition on 25% of trials, conditions not in the
recordings such as light ending at the go cue, 1.3 s photoinhibition and four-spot unilateral
photoinhibition). Those sessions are not in the dandiset, so the claims are blocked. The behaviour
recorded during the electrophysiology sessions is shown here for reference only; it is a different
set of sessions and does not decide an outcome.

In [ ]:
def perf(tr, cond, side=None, ignore_as_error=False):
    m = tr.behaving & ~tr.early_lick & (tr.cond == cond)
    if not ignore_as_error:
        m &= tr.outcome.isin(["hit", "miss"])
    if side:
        m &= tr.instruction == side
    t = tr[m]
    return (t.outcome == "hit").mean() if len(t) else np.nan, len(t)

rows = []
for s in S.values():
    if s["exp"] not in ("exp1", "exp2"):
        continue
    for c in ["control", "sample", "early_delay", "late_delay", "left1", "right1", "bilateral8"]:
        p, n = perf(s["trials"], c)
        pr, _ = perf(s["trials"], c, "right")
        if n:
            rows.append({"experiment": s["exp"], "subject": s["subject"], "cond": c, "perf": p, "perf_right": pr, "n": n})
bp = pd.DataFrame(rows)
mouse = bp.groupby(["experiment", "subject", "cond"]).apply(lambda d: pd.Series({"perf": np.average(d.perf, weights=d.n), "perf_right": np.nanmean(d.perf_right)})).reset_index()
for exp, conds in [("exp1", ["sample", "early_delay", "late_delay"]), ("exp2", ["left1", "right1", "bilateral8"])]:
    m = mouse[mouse.experiment == exp].pivot(index="subject", columns="cond", values="perf")
    print(f"{exp}: performance per mouse (fraction correct, lick-early excluded)")
    print(m.round(3).to_string())
    for c in conds:
        tp = stats.ttest_rel(m[c], m["control"]).pvalue
        ti = stats.ttest_ind(m[c], m["control"]).pvalue
        print(f"  {c}: mean {m[c].mean():.3f} against control {m['control'].mean():.3f}; paired p = {tp:.3g}, unpaired p = {ti:.3g}")

# Experiment 3 behaviour per mouse (verifier's request; reference only), and one-spot unilateral
# photoinhibition with the two hemispheres pooled per mouse (trials of left1 and right1 together).
rows = []
for s in S.values():
    if s["exp"] not in ("exp2", "exp3"):
        continue
    tr = s["trials"]
    for c in ["control", "left1", "right1", "left4", "right4", "bilateral2", "bilateral8", "one spot pooled"]:
        cc = ["left1", "right1"] if c == "one spot pooled" else [c]
        m = tr.behaving & ~tr.early_lick & tr.cond.isin(cc) & tr.outcome.isin(["hit", "miss"])
        if m.sum():
            rows.append({"experiment": s["exp"], "subject": s["subject"], "cond": c, "hits": int((tr.outcome[m] == "hit").sum()), "n": int(m.sum())})
bq = pd.DataFrame(rows).groupby(["experiment", "subject", "cond"])[["hits", "n"]].sum().reset_index()
bq["perf"] = bq.hits / bq.n
for exp in ("exp3", "exp2"):
    m = bq[bq.experiment == exp].pivot(index="subject", columns="cond", values="perf")
    nn = bq[bq.experiment == exp].groupby("cond").n.sum().to_dict()
    print(f"\n{exp}: performance per mouse (trials pooled over sessions, lick-early excluded); trials per condition {nn}")
    print(m.round(3).to_string())
    for c in [x for x in m.columns if x != "control"]:
        d = m[[c, "control"]].dropna()
        if len(d) >= 2:
            print(f"  {c}: mean {d[c].mean():.3f} against control {d['control'].mean():.3f} over {len(d)} mice; "
                  f"paired p = {stats.ttest_rel(d[c], d['control']).pvalue:.3g}, unpaired p = {stats.ttest_ind(d[c], d['control']).pvalue:.3g}")
print("\nPaper (Fig. 2b, 13 mice of the behavioural cohort): one spot 70.3%, four spots 67.7%, bilateral two spots 58.0%, eight spots 51.4%.")

## ed4-return-to-trajectory

> Neurons whose spike rates decreased during the delay epoch also recovered after unilateral
> photoinhibition, returning to the time-matched control trajectory and not decaying to the
> end-point rate.

Criterion: experiment 1; neurons with a significant decrease of the spike count in the last 400 ms
of the delay against the first 400 ms (two-tailed t-test, p < 0.05, trial types pooled); rates
normalised to the neuron's mean rate; for each condition the RMS between photoinhibition offset and
the go cue of the perturbed minus time-matched control rate and of the perturbed rate minus the
control rate at the end of the delay; readings (a) one value per neuron averaged over conditions
and trial types, (b) one per neuron, condition and trial type; reproduced when the time-matched RMS
is smaller by a paired t-test with p < 0.001 under a reading. Answered 2026-10-21 as not given by
the paper: the selection test paired (s1) or unpaired (s2), the bin of the rate time course 200 ms
(t1) or 400 ms (t2), and lick-early trials dropped (K1) or kept (K2); the outcome must hold under
all, and the count of 27 is reported against each. Neurons are the putative pyramidal neurons (convention U); trials are correct and incorrect
ones; the mean rate is over control trials from sample onset to 1.3 s after the go cue.

In [ ]:
OFF1 = {"sample": -1.6, "early_delay": -0.8, "late_delay": -0.3}
e1 = U[(U.exp == "exp1") & (U.cls == "pyr")]
ed4_rows, ed4_detail = [], {}
for early in ("drop", "keep"):
    for paired in (True, False):
        chosen = []
        for u in e1.itertuples():
            tr = S[u.sid]["trials"]
            m = L.trial_mask(tr, early) & (tr.cond == "control").values & u.observed
            t = tr[m]
            a = L.counts_in(u.spikes, t.go.values, (t.delay - t.go).values, (t.delay - t.go).values + 0.4)
            b = L.counts_in(u.spikes, t.go.values, -0.4, 0.0)
            if len(t) < 3:
                continue
            p = stats.ttest_rel(b, a).pvalue if paired else stats.ttest_ind(b, a).pvalue
            if p < 0.05 and b.mean() < a.mean():
                chosen.append(u.uid)
        for w in (0.2, 0.4):
            sub = e1[e1.uid.isin(chosen)]
            rm = L.rate_matrices(sub, S, w, early)
            tl = L.labels(w, "centre")
            j_end = int(np.argmin(abs(L.GRID_END - 0.0)))     # window ending at the go cue
            per = []
            for uid, d in rm.items():
                ctrl = d["cond"] == "control"
                so = d["so"][ctrl]
                n_all = L.counts_in(d["spikes"], d["go"][ctrl], so, np.full(ctrl.sum(), 1.3))
                mean_rate = n_all.sum() / np.sum(1.3 - so)
                if mean_rate <= 0:
                    continue
                for c, off in OFF1.items():
                    win = (tl > off) & (tl <= 0)
                    for side in (True, False):
                        pm = (d["cond"] == c) & (d["right"] == side)
                        cm = ctrl & (d["right"] == side)
                        if pm.sum() < 1 or cm.sum() < 1:
                            continue
                        rp = d["rate"][pm].mean(0) / mean_rate
                        rc = d["rate"][cm].mean(0) / mean_rate
                        per.append({"uid": uid, "cond": c, "right": side,
                                    "rms_matched": np.sqrt(np.mean((rp[win] - rc[win]) ** 2)),
                                    "rms_endpoint": np.sqrt(np.mean((rp[win] - rc[j_end]) ** 2))})
            per = pd.DataFrame(per)
            ed4_detail[(early, paired, w)] = per
            avg = per.groupby("uid")[["rms_matched", "rms_endpoint"]].mean()
            ta = stats.ttest_rel(avg.rms_matched, avg.rms_endpoint)
            tb = stats.ttest_rel(per.rms_matched, per.rms_endpoint)
            ed4_rows.append({"early_lick": early, "selection_test": "paired" if paired else "unpaired", "bin_s": w,
                             "neurons": len(chosen), "mice": sub.subject.nunique(),
                             "a_matched": avg.rms_matched.mean(), "a_endpoint": avg.rms_endpoint.mean(), "a_t": ta.statistic, "a_p": ta.pvalue,
                             "b_n": len(per), "b_matched": per.rms_matched.mean(), "b_endpoint": per.rms_endpoint.mean(), "b_t": tb.statistic, "b_p": tb.pvalue})
ed4 = pd.DataFrame(ed4_rows)
ed4["holds_a"] = (ed4.a_p < 0.001) & (ed4.a_matched < ed4.a_endpoint)
ed4["holds_b"] = (ed4.b_p < 0.001) & (ed4.b_matched < ed4.b_endpoint)
print(ed4.round(4).to_string(index=False))
print("\nPaper: 27 neurons from 6 mice (tolerance 2.7), paired t-test p < 0.001.")

**Result: not reproduced.** The selection finds 47 to 67 putative pyramidal neurons with a
significant delay-epoch decrease, from 4 mice (paper 27 from 6). Under no reading is the RMS of the
difference from the time-matched control smaller than the RMS of the difference from the end-point
rate: the two are within 0.02 of each other in every row (paired p 0.41 to 0.98 with one value per
neuron, 0.30 to 0.98 with one value per neuron, condition and trial type). This holds under all eight combinations of s1, s2, t1, t2, K1 and K2
that the answer requires, so the outcome is unchanged.

## fig1f-spike-rate-recovery

> Within 400 ms after unilateral photoinhibition, spike rates became indistinguishable from control
> in 90% of neurons; 10% retained a sustained change.

Criterion: convention R on putative pyramidal neurons of experiment 1, per condition; the fraction
of neurons with a significant change (p < 0.01) in the 400 ms window at 400 ms after
photoinhibition offset; readings (a) each of the three conditions, (b) the three pooled; window
labelled by its end (W1) or centre (W2); neurons below 1 spike/s or with fewer than 3 trials
excluded (X1) or not (X2). Reproduced when the fraction is within 0.09 to 0.11 under one
combination. Lick-early trials dropped (K1) or kept (K2); answered 2026-10-21 as not given by the
paper, the outcome must hold under both.

In [ ]:
f1_rows = []
for early in ("drop", "keep"):
    for X in ("X1", "X2"):
        res_c = {}
        for c, off in OFF1.items():
            P, info = L.rate_change_tests(e1, S, c, 0.4, early, min_rate=1 if X == "X1" else None, min_trials=3 if X == "X1" else None)
            res_c[c] = (P, info)
        for lab in ("end", "centre"):
            tl = L.labels(0.4, lab)
            tot_sig = tot_n = 0
            row = {"early_lick": early, "exclusion": X, "label": "W1 end" if lab == "end" else "W2 centre"}
            for c, off in OFF1.items():
                P, info = res_c[c]
                inc = info.included.values & ~np.isnan(P).all(1)
                j = int(np.argmin(abs(tl - (off + 0.4))))
                sig = P[inc, j] < 0.01
                row[f"n_{c}"] = int(inc.sum()); row[f"frac_{c}"] = sig.mean()
                jd = int(np.argmin(abs(tl - off)))
                row[f"during_{c}"] = (P[inc, jd] < 0.01).mean()
                tot_sig += sig.sum(); tot_n += inc.sum()
            row["frac_pooled"] = tot_sig / tot_n
            f1_rows.append(row)
f1 = pd.DataFrame(f1_rows)
f1["a_each_within"] = f1[[f"frac_{c}" for c in OFF1]].sub(0.1).abs().le(0.01 + 1e-12).all(axis=1)
f1["b_pooled_within"] = (f1.frac_pooled - 0.1).abs() <= 0.01 + 1e-12
print(f1.round(3).to_string(index=False))
print("\nPaper: n = 168, 168, 175 neurons; fraction 0.10 (tolerance 0.01). 'during_' is the fraction at the window labelled at offset.")

**Result: reproduced.** With windows labelled by their centre and no exclusion (W2, X2), the pooled
fraction of neurons with a significant change 400 ms after offset is 0.101 with lick-early trials
dropped and 0.097 with them kept, both within 0.09 to 0.11, so it holds under K1 and K2 as the answer
requires. The per-condition fractions under that
reading are 0.105, 0.113 and 0.085 (dropped), so reading (a) does not hold; with the exclusion (X1)
170 to 172 neurons are tested per condition (paper 168 to 175) and the pooled fraction is 0.112 to
0.165. During the light 27% to 72% of neurons are significant, depending on the window label. The passing reading is the
one without the exclusion, and it tests 248 neurons per condition, against the paper's 168 to 175;
the reading with the exclusion (X1) comes close to the paper's counts (170 to 172) and fails. The
criterion allows either, so the outcome is reproduced, but it rests on the reading whose neuron
count differs from the paper's.

## fig1g-selectivity-recovery

> After unilateral photoinhibition ALM neurons recovered more than 80% of their control selectivity
> within about 0.5 s; recovery times were 373 ms (sample), 510 ms (early delay) and 327 ms (late
> delay).

Criterion: convention D on experiment 1 (selective neurons tested on more than 3 trials in every
condition; preferred trial type from 10 random control trials, redrawn 20 times and averaged;
selectivity on correct and incorrect trials grouped by instruction); recovery time is the first
window after photoinhibition offset where the population selectivity on photoinhibition trials
reaches 80% of the time-matched control selectivity; bins of 200 ms (B1) or 400 ms (B2) in 50 ms
steps. Reproduced when all three times are within the paper's s.e.m. under one bin reading;
partially for two of three. The selective neurons are those of selective-fraction under each of
its readings. Answered 2026-10-21, none given by the paper: lick-early trials (K1 dropped, K2
kept); the window of the preference counts (D1 whole trial, D2 delay epoch, D3 each bin); the label
of a bin (L1 centre, L2 end); the end of the search (E1 the go cue, E2 1.6 s after it, the end of
the windows; the paper's late-delay time of 327 ms after an offset 0.3 s before the go cue falls
after the go cue). The outcome is the one all of them give, else `methods_underspecified`. Trials
are counted per condition (N1, the legend's words); per condition and trial type (N2) is a variant.
The measured time is that of the population of all included neurons; the bootstrap (200 resamples
of neurons) mean and s.d. are shown beside it.

In [ ]:
t0 = time.time()
rec1 = pd.concat([L.recovery_readings(e1, S, OFF1, SEL[early], early)[0] for early in ("drop", "keep")], ignore_index=True)
rec1 = rec1.rename(columns={"early": "early_lick"})
exp1g = {"sample": (373, 260), "early_delay": (510, 218), "late_delay": (327, 112)}
for c, (v, tol) in exp1g.items():
    rec1[c + "_ok"] = (rec1[c] - v).abs() <= tol
rec1["n_ok"] = (rec1.n - 55).abs() <= 5.5
rec1["times_ok"] = rec1[[c + "_ok" for c in exp1g]].sum(axis=1)
print(f"{len(rec1)} reading combinations in {time.time() - t0:.0f} s")
print("pref: trial = D1, delay = D2, bin = D3; label: centre = L1, end = L2; search: E1 to the go cue, E2 to 1.6 s after it.")
print("Trials counted per condition (N1, decides the outcome):")
show = ["early_lick", "pref", "label", "search", "bin", "S", "n", "sample", "early_delay", "late_delay", "sample_boot", "sample_sd",
        "early_delay_boot", "early_delay_sd", "late_delay_boot", "late_delay_sd", "times_ok"]
print(rec1[rec1.trials_rule == "per condition"][show].round(0).to_string(index=False))


In [ ]:
def outcome_1g(d):
    return "reproduced" if (d.times_ok == 3).any() else "partially" if (d.times_ok == 2).any() else "not"
g = rec1.groupby(["trials_rule", "early_lick", "pref", "label", "search"])
summary1g = g.apply(lambda d: pd.Series({"combos": len(d), "all_three": int((d.times_ok == 3).sum()), "two": int((d.times_ok == 2).sum()),
                                        "n_range": f"{d.n.min()}-{d.n.max()}", "outcome_here": outcome_1g(d)})).reset_index()
print("Per combination of the method details (K, D, L, E); any S reading and bin reading (B1, B2) may match:")
print(summary1g[summary1g.trials_rule == "per condition"].drop(columns="trials_rule").to_string(index=False))
for rule in ("per condition", "per type"):
    oc = summary1g[summary1g.trials_rule == rule].outcome_here.value_counts().to_dict()
    print(f"\nOutcomes over the {int((summary1g.trials_rule == rule).sum())} method combinations, trials {rule}"
          f" ({'N1, decides' if rule == 'per condition' else 'N2, variant'}): {oc}")
print("Included neurons, N1:", rec1[rec1.trials_rule == "per condition"].n.min(), "to", rec1[rec1.trials_rule == "per condition"].n.max(),
      "; N2:", rec1[rec1.trials_rule == "per type"].n.min(), "to", rec1[rec1.trials_rule == "per type"].n.max(), "; paper 55 (tolerance 5.5)")


In [ ]:
# Population sizes (verifier's request, reference only). For the included neurons of convention D
# (selective under reading R1-O1-student, more than 3 trials in every condition, N1), the fewest trials
# a neuron has in any condition, and the epochs in which it is selective. This does not change the
# selection, which the claims file fixes; it shows which further selections would bring the counts
# near the paper's 55, 143 and 40.
def included_neurons(units, conds, early, skey="R1-O1-student"):
    st = tests[(early, skey)].set_index("uid")
    rows = []
    for u in units[units.uid.isin(SEL[early][skey])].itertuples():
        tr = S[u.sid]["trials"]
        m = L.trial_mask(tr, early) & u.observed
        right = (tr.instruction == "right").values
        n = {}
        ok = True
        for c in ("control",) + tuple(conds):
            mc = m & (tr.cond == c).values
            n[c] = int(mc.sum())
            ok &= mc.sum() > 3 and (mc & right).sum() > 0 and (mc & ~right).sum() > 0
        if ok:
            p = st.loc[u.uid]
            rows.append({"uid": u.uid, "subject": u.subject, "min_trials": min(n.values()),
                         "min_trials_per_type": min(min(int((m & (tr.cond == c).values & right).sum()), int((m & (tr.cond == c).values & ~right).sum()))
                                                    for c in ("control",) + tuple(conds)),
                         "sample": p["sample"] < 0.05, "delay": p["delay"] < 0.05, "response": p["response"] < 0.05})
    return pd.DataFrame(rows)
prow = []
for name, units_, conds, paper in (("exp1 (fig1g)", e1, list(OFF1), 55), ("exp2 (fig2e)", U[(U.exp == "exp2") & (U.cls == "pyr")], ["left1", "right1", "bilateral8"], 143),
                                   ("exp3 (ed7)", U[(U.exp == "exp3") & (U.cls == "pyr")], ["left4", "right4", "bilateral2"], 40)):
    for early in ("drop", "keep"):
        d = included_neurons(units_, conds, early)
        prow.append({"experiment": name, "early_lick": early, "paper": paper, "included": len(d), "mice": d.subject.nunique(),
                     "min_trials_q10_q50_q90": "/".join(str(int(x)) for x in d.min_trials.quantile([0.1, 0.5, 0.9])),
                     "min_trials_gt_10": int((d.min_trials > 10).sum()), "min_trials_gt_20": int((d.min_trials > 20).sum()),
                     "per_type_gt_3": int((d.min_trials_per_type > 3).sum()), "per_type_gt_5": int((d.min_trials_per_type > 5).sum()),
                     "sel_sample": int(d["sample"].sum()), "sel_delay": int(d.delay.sum()), "sel_response": int(d.response.sum()),
                     "delay_only": int((d.delay & ~d["sample"] & ~d.response).sum()), "sample_or_delay": int((d["sample"] | d.delay).sum()),
                     "sample_and_delay": int((d["sample"] & d.delay).sum())})
print(pd.DataFrame(prow).to_string(index=False))

**Result: blocked (methods_underspecified).** The answer confirmed that the paper gives none of
the details asked about, and the claims file now requires one outcome over all of them (K1, K2; D1
to D3; L1, L2; E1, E2), with trials counted per condition (N1). They give different outcomes: of the
24 combinations, 12 have readings with all three recovery times within tolerance, 9 have readings
with two and 3 have none. With lick-early trials dropped, all three are within tolerance under
every combination whose search runs past the go cue (E2), and under E1 with centre labels; E1 with
end labels reaches two. With lick-early trials kept, three combinations reach none (end labels with
whole-trial preference under E1 and E2, and with delay-epoch preference under E1). Counting trials
per condition and trial type (N2, a variant) gives 11 combinations with all three, 12 with two and
1 with none. Every reading includes 138 to 173 neurons, against the paper's 55.

Under E1 with end labels the late-delay time is never found (NaN in all 96 readings): no bin up to
the go cue reaches 80% of control. The paper's 327 ms falls 27 ms after the go cue for this
condition (offset 0.3 s before it), so a search ending at the go cue can give at most 300 ms, which
is inside the tolerance (215 to 439 ms); a question asks the reader whether E1 is meant for this
condition. Population sizes (reference only): of the 162 neurons included with K1 (R1-O1-Student),
the fewest trials in any condition has a median of 27 (10th percentile 11); 118 have more than 20
trials in every condition, 73 are selective in the delay epoch, 95 in the sample or delay epoch and
15 in the delay epoch only. None of these selections gives the paper's 55, and the same holds for
experiment 2 (286 included, 160 delay-selective, 113 with more than 20 trials; paper 143) and
experiment 3 (58, 34, 46; paper 40).

## ed2b-silencing-strength

> Ipsilateral and bilateral photoinhibition reduced the activity of putative pyramidal neurons by
> more than 90%, similarly in VGAT-ChR2-EYFP and PV x ReaChR mice; contralateral photoinhibition
> had little effect on the average rate.

Criterion: per neuron (mean rate at least 1 spike/s), the rate during photoinhibition over the
control rate in the same window; window (a) the whole photostimulus before the ramp, (b) the last
400 ms of photoinhibition. Reproduced when the mean normalised rate is below 0.1 for ipsilateral
photoinhibition in both mouse lines (experiment 2) and for bilateral photoinhibition, and
contralateral photoinhibition leaves it above 0.8. Judged on experiment 2; experiments 1 and 3 are
shown. Neurons need 2 trials in the condition and in control (a standard choice); trials are
correct and incorrect ones; lick-early trials dropped (K1) or kept (K2), answered 2026-10-21 as not
given by the paper, so the outcome must hold under both.

In [ ]:
CONDS = {"exp1": {"sample": "ipsilateral", "early_delay": "ipsilateral", "late_delay": "ipsilateral"},
         "exp2": {"left1": "ipsilateral", "right1": "contralateral", "bilateral8": "bilateral"},
         "exp3": {"left4": "ipsilateral", "right4": "contralateral", "bilateral2": "bilateral"}}
RAMP = {"exp1": 0.1, "exp2": 0.2, "exp3": 0.2}
rows = []
for early in ("drop", "keep"):
    for exp, cmap in CONDS.items():
        uu = U[(U.exp == exp) & (U.cls == "pyr")]
        for u in uu.itertuples():
            tr = S[u.sid]["trials"]
            m = L.trial_mask(tr, early) & u.observed
            k = m & (tr.cond == "control").values
            so = (tr["sample"] - tr.go).values[k]
            n_all = L.counts_in(u.spikes, tr.go.values[k], so, np.full(k.sum(), 1.3))
            mean_rate = n_all.sum() / np.sum(1.3 - so) if k.sum() else 0
            if mean_rate < 1:
                continue
            for c, kind in cmap.items():
                p = m & (tr.cond == c).values
                if p.sum() < 2 or k.sum() < 2:
                    continue
                on, off = np.median(tr.ps_on_re_go.values[p & ~tr.early_lick.values]), np.median(tr.ps_off_re_go.values[p & ~tr.early_lick.values])
                for wname, (a, b) in {"a": (on, off - RAMP[exp]), "b": (off - 0.4, off)}.items():
                    cp = L.counts_in(u.spikes, tr.go.values[p], a, b).mean()
                    cc = L.counts_in(u.spikes, tr.go.values[k], a, b).mean()
                    rows.append({"early_lick": early, "experiment": exp, "cond": c, "kind": kind, "window": wname,
                                 "genotype": u.genotype, "uid": u.uid, "norm": cp / cc if cc > 0 else np.nan})
nr = pd.DataFrame(rows)
tab = nr.groupby(["early_lick", "window", "experiment", "cond", "kind"]).norm.agg(["size", "mean", "sem"]).round(3)
print(tab.to_string())
print("\nExperiment 2, ipsilateral, by mouse line:")
print(nr[(nr.experiment == "exp2") & (nr.kind == "ipsilateral")].groupby(["early_lick", "window", "genotype"]).norm.agg(["size", "mean", "sem"]).round(3).to_string())
print("\nLegend counts: experiment 1 n = 117, 110, 109; experiment 2 n = 300, 294, 301; experiment 3 n = 52, 52, 102;"
      " experiment 2 ipsilateral 94 VGAT and 201 PV neurons.")
judge = []
for (early, wname), d in nr[nr.experiment == "exp2"].groupby(["early_lick", "window"]):
    ip = d[d.kind == "ipsilateral"].groupby("genotype").norm.mean()
    judge.append({"early_lick": early, "window": wname, "ipsi_VGAT": ip.get("VGAT"), "ipsi_PV": ip.get("PV"),
                  "bilateral": d[d.kind == "bilateral"].norm.mean(), "contralateral": d[d.kind == "contralateral"].norm.mean()})
judge = pd.DataFrame(judge)
judge["holds"] = (judge.ipsi_VGAT < 0.1) & (judge.ipsi_PV < 0.1) & (judge.bilateral < 0.1) & (judge.contralateral > 0.8)
print("\n", judge.round(3).to_string(index=False))

**Result: not reproduced.** Photoinhibition silences most activity, but not by more than 90%
in every case. With lick-early trials dropped (K1), the normalised rate in experiment 2 is 0.077 to
0.097 for bilateral and 0.94 to 1.00 for contralateral photoinhibition, and 0.035 to 0.052 for
ipsilateral photoinhibition in PV x ReaChR mice, but 0.177 to 0.245 in VGAT-ChR2-EYFP mice: three of
the four parts hold. With lick-early trials kept (K2) only the contralateral part holds (1.01 to
1.04); ipsilateral photoinhibition leaves 0.139 to 0.140 (PV) and 0.282 to 0.356 (VGAT), and
bilateral 0.169 to 0.196. The outcome must be the one K1 and K2 both give, and under K2 one part of
four holds, so the claim is not reproduced (the first round gave partial; the rule is in the
Summary). Part of the K2 rise comes from lick-early trials whose light falls before its logged
time, 6.5% to 7.5% of the experiment-2 photoinhibition trials (data check), which enter the
photoinhibition average without light in the window. Experiment 1 gives 0.037 to 0.098 (K1). Neuron
counts are 172, 299 and 55 to 110 per condition, against 109 to 117, 294 to 301 and 52 to 102.
Neurons need 2 trials in the condition and in control, a standard choice.

## ed2e-no-rebound

> No rebound excitation was detected on average after photoinhibition offset; a small fraction of
> neurons showed rebound, balanced by weak sustained inhibition in a larger fraction.

Criterion: experiment 2, photoinhibition 0.8 s at delay onset, lick right trials only, the six
400 ms windows of the legend; per neuron control against photoinhibition by two-tailed t-test,
p < 0.01. Reproduced when, in the first 400 ms and 400 to 800 ms after photoinhibition
(ipsilateral), the population mean rate is not higher than control (paired t-test across neurons,
p > 0.05, a reading) and fewer neurons have a significant increase than a significant decrease.
Neurons: putative pyramidal with at least 2 lick right trials in each condition (a standard choice);
lick-early trials dropped (K1) or kept (K2), answered 2026-10-21 as not given by the paper, so
the outcome must hold under both.

In [ ]:
WIN = {"last 400 ms sample": (-2.1, -1.7), "first 400 ms light": (-1.7, -1.3), "last 400 ms light": (-1.3, -0.9),
       "first 400 ms after": (-0.9, -0.5), "400-800 ms after": (-0.5, -0.1), "first 400 ms response": (0.0, 0.4)}
e2 = U[(U.exp == "exp2") & (U.cls == "pyr")]
rows = []
for early in ("drop", "keep"):
    for c in ("left1", "right1", "bilateral8"):
        for u in e2.itertuples():
            tr = S[u.sid]["trials"]
            m = L.trial_mask(tr, early) & u.observed & (tr.instruction == "right").values
            k, p = m & (tr.cond == "control").values, m & (tr.cond == c).values
            if k.sum() < 2 or p.sum() < 2:
                continue
            for wn, (a, b) in WIN.items():
                cc = L.counts_in(u.spikes, tr.go.values[k], a, b) / (b - a)
                cp = L.counts_in(u.spikes, tr.go.values[p], a, b) / (b - a)
                pv = L.ttest(cp, cc)
                rows.append({"early_lick": early, "cond": c, "window": wn, "uid": u.uid, "ctrl": cc.mean(), "pi": cp.mean(),
                             "up": pv < 0.01 and cp.mean() > cc.mean(), "down": pv < 0.01 and cp.mean() < cc.mean()})
rb = pd.DataFrame(rows)
def summ(d):
    t = stats.ttest_rel(d.pi, d.ctrl)
    return pd.Series({"neurons": len(d), "ctrl_mean": d.ctrl.mean(), "pi_mean": d.pi.mean(), "paired_t": t.statistic, "paired_p": t.pvalue,
                      "n_up": int(d.up.sum()), "n_down": int(d.down.sum())})
rbs = rb.groupby(["early_lick", "cond", "window"], sort=False).apply(summ).reset_index()
print(rbs.round(4).to_string(index=False))
j = rbs[(rbs.cond == "left1") & rbs.window.isin(["first 400 ms after", "400-800 ms after"])].copy()
j["not_higher"] = ~((j.pi_mean > j.ctrl_mean) & (j.paired_p < 0.05))
j["up_lt_down"] = j.n_up < j.n_down
print("\nJudged windows (ipsilateral):")
print(j[["early_lick", "window", "pi_mean", "ctrl_mean", "paired_p", "n_up", "n_down", "not_higher", "up_lt_down"]].round(4).to_string(index=False))

**Result: not reproduced.** After ipsilateral photoinhibition the population mean rate is not higher
than control in either window (paired p 0.93 and 0.22 with lick-early trials dropped, 0.70 and 0.06
kept), but more neurons show a significant increase than a significant decrease: 27 against 7 in
the first 400 ms and 30 against 3 in 400 to 800 ms (dropped; 31 against 10 and 31 against 4 kept),
the reverse of the legend's balance. After bilateral photoinhibition the population mean rises (3.84
against 3.21 spikes/s, p = 0.002, in 400 to 800 ms). The counts fail under K1 and K2 alike, so
the answer leaves the outcome unchanged.

## ed6a-bilateral-inconsistent-bias

> Bilateral photoinhibition biased movements inconsistently across mice and sessions, toward lick
> left in some and lick right in others.

Criterion: per session of the bilateral experiment, bias = (lick right minus lick left performance)
under bilateral photoinhibition minus the same in control; reproduced when, among sessions with a
bias significant by Fisher's exact test (p < 0.05), both signs occur, in at least two mice. Which
2 x 2 table enters Fisher's test was answered 2026-10-21 as not given by the paper; readings (f1,
printed as p_i) bilateral trials, trial type by correct; (f2, printed as p_ii) licked side by
condition (control against bilateral); the outcome must hold under both. Experiment 2 sessions
(bilateral photoinhibition of four spots per side); performance excludes lick-early trials
(convention P) and no-response trials (shown with them counted as errors as a variant).
Experiment 3 (one spot per side) is shown as a variant.

In [ ]:
def session_bias(tr, bil, ignore_as_error=False):
    outs = ("hit", "miss", "ignore") if ignore_as_error else ("hit", "miss")
    m = tr.behaving & ~tr.early_lick & tr.outcome.isin(outs)
    out = {}
    for c in ("control", bil):
        t = tr[m & (tr.cond == c)]
        r, l = t[t.instruction == "right"], t[t.instruction == "left"]
        out[c] = dict(pr=(r.outcome == "hit").mean(), pl=(l.outcome == "hit").mean(),
                      rh=int((r.outcome == "hit").sum()), rn=len(r), lh=int((l.outcome == "hit").sum()), ln=len(l),
                      lick_right=int(((t.instruction == "right") & (t.outcome == "hit")).sum() + ((t.instruction == "left") & (t.outcome != "hit")).sum()),
                      n=len(t))
    b, c = out[bil], out["control"]
    bias = (b["pr"] - b["pl"]) - (c["pr"] - c["pl"])
    p_i = stats.fisher_exact([[b["rh"], b["rn"] - b["rh"]], [b["lh"], b["ln"] - b["lh"]]]).pvalue
    p_ii = stats.fisher_exact([[b["lick_right"], b["n"] - b["lick_right"]], [c["lick_right"], c["n"] - c["lick_right"]]]).pvalue
    return bias, p_i, p_ii, b["n"]

rows = []
for s in S.values():
    for exp, bil in (("exp2", "bilateral8"), ("exp3", "bilateral2")):
        if s["exp"] != exp or not s["trials"].behaving.any():
            continue
        for ign in (False, True):
            bias, p_i, p_ii, n = session_bias(s["trials"], bil, ign)
            rows.append({"experiment": exp, "sid": s["sid"], "subject": s["subject"], "ignore_as_error": ign, "bilateral_trials": n,
                         "bias": bias, "p_i": p_i, "p_ii": p_ii})
b6 = pd.DataFrame(rows)
BIAS = b6[(b6.experiment == "exp2") & ~b6.ignore_as_error].set_index("sid")
print(b6[~b6.ignore_as_error].round(4).to_string(index=False))
for (exp, ign), d in b6.groupby(["experiment", "ignore_as_error"]):
    for col in ("p_i", "p_ii"):
        sig = d[d[col] < 0.05]
        pos, neg = sig[sig.bias > 0], sig[sig.bias < 0]
        print(f"{exp}, no-response as error {ign}, Fisher reading {col}: {len(sig)} significant sessions; "
              f"lick right bias {len(pos)} sessions in {pos.subject.nunique()} mice, lick left bias {len(neg)} sessions in {neg.subject.nunique()} mice; "
              f"both signs, in at least two mice in all: {len(pos) > 0 and len(neg) > 0 and sig.subject.nunique() >= 2}; "
              f"each sign in at least two mice: {pos.subject.nunique() >= 2 and neg.subject.nunique() >= 2}")

**Result: reproduced.** In experiment 2, 12 sessions (Fisher reading f1) or 13 (f2) have a
significant bias; 7 or 8 of them are biased toward lick right, in 4 mice, and 5 toward lick left, in
4 mice, with no-response trials left out or counted as errors. Both signs therefore occur, each in
at least two mice. It holds under f1 and f2, as the answer requires. Experiment 3 (one spot per side) shows few significant sessions (2 or 3).

## fig2e-selectivity-recovery-times

> Selectivity recovered to 80% of control 538 ms after ipsilateral and 192 ms after contralateral
> photoinhibition, and did not recover after bilateral photoinhibition.

Criterion: convention D on experiment 2 (ipsilateral one spot, contralateral one spot, bilateral
four spots per side; light 0.8 s from delay onset, offset 0.9 s before the go cue). Bilateral, no
recovery means photoinhibition selectivity below 80% of control up to the go cue. Reproduced when
both times are within tolerance and bilateral does not recover, under one bin reading. The method details
answered 2026-10-21 (K, D, L, E) are run as for fig1g, and the outcome must be the same under all;
trials per condition (N1) decide, per condition and trial type (N2) is a variant.

In [ ]:
OFF2 = {"left1": -0.9, "right1": -0.9, "bilateral8": -0.9}
t0 = time.time()
rec2, courses2 = {}, {}
for early in ("drop", "keep"):
    rec2[early], courses2[early] = L.recovery_readings(e2, S, OFF2, SEL[early], early, no_recovery=("bilateral8",))
rec2 = pd.concat(rec2.values(), ignore_index=True).rename(columns={"early": "early_lick"})
rec2["ipsi_ok"] = (rec2.left1 - 538).abs() <= 178
rec2["contra_ok"] = (rec2.right1 - 192).abs() <= 114
rec2["bilateral_no_recovery"] = ~rec2.bilateral8_recovers_by_go
rec2["holds"] = rec2.ipsi_ok & rec2.contra_ok & rec2.bilateral_no_recovery
print(f"{len(rec2)} reading combinations in {time.time() - t0:.0f} s")
print("Trials counted per condition (N1, decides the outcome):")
print(rec2[rec2.trials_rule == "per condition"][["early_lick", "pref", "label", "search", "bin", "S", "n", "left1", "left1_boot", "left1_sd",
            "right1", "right1_boot", "right1_sd", "bilateral8_max_ratio_to_go", "holds"]].round(2).to_string(index=False))


In [ ]:
s2e = rec2.groupby(["trials_rule", "early_lick", "pref", "label", "search"]).apply(
    lambda d: pd.Series({"combos": len(d), "holds": int(d.holds.sum()), "ipsi_ok": int(d.ipsi_ok.sum()), "contra_ok": int(d.contra_ok.sum()),
                         "bilateral_no_recovery": int(d.bilateral_no_recovery.sum()), "n_range": f"{d.n.min()}-{d.n.max()}",
                         "left1_ms": f"{d.left1.min():.0f}-{d.left1.max():.0f}", "left1_nan": int(d.left1.isna().sum()),
                         "right1_ms": f"{d.right1.min():.0f}-{d.right1.max():.0f}"})).reset_index()
s2e["outcome_here"] = np.where(s2e.holds > 0, "reproduced", "not")
print(s2e[s2e.trials_rule == "per condition"].drop(columns="trials_rule").to_string(index=False))
for rule in ("per condition", "per type"):
    print(f"Outcomes over the method combinations, trials {rule}:", s2e[s2e.trials_rule == rule].outcome_here.value_counts().to_dict())
d1 = rec2[rec2.trials_rule == "per condition"]
print("\nN1: ipsilateral time", d1.left1.min(), "to", d1.left1.max(), "ms (no recovery in", int(d1.left1.isna().sum()), "of", len(d1),
      "); contralateral", d1.right1.min(), "to", d1.right1.max(), "ms; bilateral below 80% up to the go cue in",
      int(d1.bilateral_no_recovery.sum()), "of", len(d1))
print("Included neurons, N1:", d1.n.min(), "to", d1.n.max(), "; N2:", rec2[rec2.trials_rule == "per type"].n.min(), "to",
      rec2[rec2.trials_rule == "per type"].n.max(), "; paper 143 (tolerance 14.3)")


**Result: not reproduced.** Under all 24 combinations of the method details (K1, K2; D1 to D3; L1,
L2; E1, E2), with trials per condition (N1), no reading puts both times within tolerance, so the
outcome is the same under all and the answers leave it unchanged. With the search past the go cue
(E2), ipsilateral selectivity reaches 80% of control 1000 to 2400 ms after offset, after the go cue,
or not at all (20 of 192 readings); with the search ending at the go cue (E1) it never does. The
paper gives 538 ms. Contralateral selectivity reaches 80% of control 200 to 1300 ms after offset
(paper 192 ms), within tolerance under some readings. Bilateral selectivity stays below 80% of
control up to the go cue under all 384 readings. 260 to 293 neurons are included with trials per
condition (221 to 292 per condition and trial type), against the paper's 143.

## fig2-recordings-recovery

> After ipsilateral (one spot) photoinhibition spike rate (more than 90%) and selectivity (more than
> 80%) recovered within 600 ms; contralateral photoinhibition hardly affected activity; after
> bilateral photoinhibition spike rates recovered on average but selectivity did not.

Criterion: experiment 2, at 600 ms after photoinhibition offset (0.3 s before the go cue). Spike
rate recovery (a) fewer than 10% of neurons with a significant change (convention R), (b)
population mean rate at least 90% of control. Selectivity (convention D) at least 80% of control.
Contralateral: fewer than 10% of neurons significant in every window during and after
photoinhibition. Bilateral: rate recovered as in (a) or (b), and selectivity below 80% of control
at the go cue. Reproduced when all hold under one combination of the claim's readings (R: W1, W2,
X1, X2; D: bins and the S readings). The method details answered 2026-10-21 (K1, K2; D1 to D3; L1,
L2) must give the same outcome; trials are counted per condition (N1). The search end (E) does not
enter, as no recovery time is read.

In [ ]:
R2 = {}
for early in ("drop", "keep"):
    for X in ("X1", "X2"):
        for c in OFF2:
            R2[(early, X, c)] = L.rate_change_tests(e2, S, c, 0.4, early, min_rate=1 if X == "X1" else None, min_trials=3 if X == "X1" else None)
rate_rows = []
for (early, X, c), (P, info) in R2.items():
    inc = info.included.values & ~np.isnan(P).all(1)
    for lab in ("end", "centre"):
        tl = L.labels(0.4, lab)
        j = int(np.argmin(abs(tl - (-0.9 + 0.6))))
        during_after = (tl >= -1.7 + 0.2) & (tl <= 0) if lab == "centre" else (tl >= -1.7 + 0.4) & (tl <= 0)
        frac_t = np.nanmean(P[inc] < 0.01, axis=0)
        rate_rows.append({"early_lick": early, "exclusion": X, "cond": c, "label": lab, "n": int(inc.sum()),
                          "frac_sig_600ms": frac_t[j], "max_frac_sig_during_after": frac_t[during_after].max()})
rr = pd.DataFrame(rate_rows)
# population mean rate at 600 ms after offset, photoinhibition over control (same windows)
pop_rows = []
for early in ("drop", "keep"):
    for lab in ("end", "centre"):
        tl = L.labels(0.4, lab); j = int(np.argmin(abs(tl - (-0.3))))
        for c in OFF2:
            num = den = 0.0
            for u in e2.itertuples():
                tr = S[u.sid]["trials"]; m = L.trial_mask(tr, early) & u.observed
                k, p = m & (tr.cond == "control").values, m & (tr.cond == c).values
                if k.sum() < 2 or p.sum() < 2:
                    continue
                e = L.GRID_END[j]
                num += L.counts_in(u.spikes, tr.go.values[p], e - 0.4, e).mean(); den += L.counts_in(u.spikes, tr.go.values[k], e - 0.4, e).mean()
            pop_rows.append({"early_lick": early, "label": lab, "cond": c, "pop_rate_ratio_600ms": num / den})
pr = pd.DataFrame(pop_rows)
print(rr.round(3).to_string(index=False))
print(pr.round(3).to_string(index=False))

In [ ]:
# selectivity ratio (photoinhibition over control) at 600 ms after offset and at the go cue
sel_rows = []
for early in ("drop", "keep"):
    for key, (tl, cm) in courses2[early].items():
        w, pw, pt, sk, lab = key
        j600 = int(np.argmin(abs(tl - (-0.3)))); jgo = int(np.argmin(abs(tl - 0.0)))
        sel_rows.append({"early_lick": early, "bin": w, "pref": pw, "trials_rule": "per type" if pt else "per condition", "S": sk, "label": lab,
                         "ipsi_sel_ratio_600ms": cm["left1"][j600] / cm["control"][j600],
                         "bilateral_sel_ratio_go": cm["bilateral8"][jgo] / cm["control"][jgo]})
sr = pd.DataFrame(sel_rows)
# rate parts: claim readings W (window label of convention R) and X; selectivity parts: claim readings B and S,
# method details K, D and L (the label of the selectivity bins), N1 deciding
rate_parts = []
for early in ("drop", "keep"):
    for W in ("end", "centre"):
        for X in ("X1", "X2"):
            r = rr[(rr.early_lick == early) & (rr.label == W) & (rr.exclusion == X)].set_index("cond")
            p = pr[(pr.early_lick == early) & (pr.label == W)].set_index("cond")
            rate_parts.append({"early_lick": early, "W": W, "X": X,
                               "ipsi_rate": (r.loc["left1", "frac_sig_600ms"] < 0.10) or (p.loc["left1", "pop_rate_ratio_600ms"] >= 0.9),
                               "bilateral_rate": (r.loc["bilateral8", "frac_sig_600ms"] < 0.10) or (p.loc["bilateral8", "pop_rate_ratio_600ms"] >= 0.9),
                               "contra_hardly_affected": r.loc["right1", "max_frac_sig_during_after"] < 0.10})
rate_parts = pd.DataFrame(rate_parts)
print(rate_parts.to_string(index=False))
out = []
for rule in ("per condition", "per type"):
    for (early, pw, lab), d in sr[sr.trials_rule == rule].groupby(["early_lick", "pref", "label"]):
        rp = rate_parts[rate_parts.early_lick == early]
        rates_all = (rp.ipsi_rate & rp.bilateral_rate & rp.contra_hardly_affected).any()
        sel_ok = (d.ipsi_sel_ratio_600ms >= 0.8) & (d.bilateral_sel_ratio_go < 0.8)
        parts = {"ipsi_rate": rp.ipsi_rate.any(), "bilateral_rate": rp.bilateral_rate.any(), "contra": rp.contra_hardly_affected.any(),
                 "ipsi_sel": (d.ipsi_sel_ratio_600ms >= 0.8).any(), "bilateral_sel": (d.bilateral_sel_ratio_go < 0.8).any()}
        holds = bool(rates_all and sel_ok.any())
        out.append({"trials_rule": rule, "early_lick": early, "pref": pw, "label": lab, **parts,
                    "ipsi_sel_ratio_600ms": f"{d.ipsi_sel_ratio_600ms.min():.2f}-{d.ipsi_sel_ratio_600ms.max():.2f}",
                    "bilateral_sel_ratio_go": f"{d.bilateral_sel_ratio_go.min():.2f}-{d.bilateral_sel_ratio_go.max():.2f}",
                    "outcome_here": "reproduced" if holds else "partially" if any(parts.values()) else "not"})
o2 = pd.DataFrame(out)
print(o2[o2.trials_rule == "per condition"].drop(columns="trials_rule").to_string(index=False))
for rule in ("per condition", "per type"):
    print(f"Outcomes over the method combinations (K, D, L), trials {rule}:", o2[o2.trials_rule == rule].outcome_here.value_counts().to_dict())
s1 = sr[sr.trials_rule == "per condition"]
print("\nN1: ipsilateral selectivity ratio at 600 ms", s1.ipsi_sel_ratio_600ms.min().round(2), "to", s1.ipsi_sel_ratio_600ms.max().round(2),
      "; bilateral selectivity ratio at the go cue", s1.bilateral_sel_ratio_go.min().round(2), "to", s1.bilateral_sel_ratio_go.max().round(2))


**Result: partially reproduced.** The outcome is the same under all 12 combinations of the method
details now required (K1, K2; D1 to D3; L1, L2), with trials per condition. Spike rates recover:
600 ms after offset 8% to 10% of neurons differ from control after ipsilateral photoinhibition, and
the population rate is 1.00 to 1.05 of control (1.16 to 1.23 after bilateral photoinhibition).
Selectivity after bilateral photoinhibition stays low, 0.22 to 0.33 of control at the go cue. Two
parts fail under every combination: ipsilateral selectivity at 600 ms is 0.45 to 0.72 of control,
not 0.8 or more; and contralateral photoinhibition changes the rate of 15% to 19% of neurons in
some window during or after the light, not fewer than 10%.

## Activity space (convention A)

The claims on the coding direction use the sessions of the bilateral experiment with more than 5
neurons recorded simultaneously for more than 5 trials per condition (control, ipsilateral,
contralateral, bilateral). Here the neurons are putative pyramidal (convention U), observed in every
analysis trial, with the training set a random half (H1) or third (H2) of the correct control trials
(the claims file's readings). The ipsilateral trials used to build modes 2 and 3 are split with the
same fraction. One random split per session (seeded by the session) is used; the spread over 10
seeds is shown as a variant. Answered 2026-10-21: the paper states no selection beyond its rule, so
the rule is followed as written, with trials counted per condition (T1) for the outcome and per
condition and trial type (T2) as a variant, and the session and neuron counts are reported against
the paper's 16 sessions with 6 to 20 neurons (mean 11). Lick-early trials are dropped (K1) or kept
(K2), and the outcome must hold under both.

In [ ]:
CONDS_A = ["control", "left1", "right1", "bilateral8"]
A = {}
t0 = time.time()
for per_type in (False, True):
    for early in ("drop", "keep"):
        sess = L.a_sessions(U, S, "exp2", CONDS_A, early, per_type)
        for frac, hname in ((0.5, "H1"), (1 / 3, "H2")):
            res = {}
            for sid, uids in sess.items():
                X, t = L.session_counts(U, S, sid, uids, early)
                rng = np.random.default_rng(int(sid.replace("_", "")) % 2 ** 32)
                res[sid] = L.activity_session(X, t, frac, CONDS_A, "left1", rng)
            A[("per type" if per_type else "per condition", early, hname)] = res
print(f"convention A on {len(A)} readings in {time.time() - t0:.0f} s")
cnt = pd.DataFrame([{"trials_rule": k[0], "early_lick": k[1], "H": k[2], "sessions": len(v), "mice": len({s.split('_')[0] for s in v}),
                     "neurons_min": min(r["n_neurons"] for r in v.values()), "neurons_max": max(r["n_neurons"] for r in v.values()),
                     "neurons_mean": np.mean([r["n_neurons"] for r in v.values()])} for k, v in A.items()])
print(cnt.round(1).to_string(index=False))
print("trials_rule: per condition = T1 (decides), per type = T2 (variant). Paper: 16 sessions of 7 mice, 6-20 neurons (mean 11).")

def by_k(df, col, extra=()):
    """Whether `col` holds under some claim reading (H, and `extra`), per lick-early reading, trials per condition (T1)."""
    d = df[df.trials_rule == "per condition"]
    return d.groupby("early_lick")[col].any().to_dict()

## cd-stable-over-epochs

> The direction of the lick right minus lick left difference vector was stable over the sample and
> delay epochs (correlation 0.61 between late sample and late delay).

Criterion: late sample and late delay are the last 400 ms window of each epoch; Pearson correlation
of the two w vectors per session, mean over the sessions; reproduced when within the paper's s.e.m.
(0.56 to 0.66). Convention A says that w comes from correct control trials and that cd, the
average of w, uses a subset of them; which trials give the w whose correlation is reported is not
fixed. Both are run, all correct control trials and the training subset (H1 or H2) that builds cd,
and the outcome is the one both give, else `methods_underspecified` (asked as a question after the
first verification round).

In [ ]:
rows = []
for k, v in A.items():
    for which in ("all", "train"):
        x = np.array([r[f"w_corr_{which}"] for r in v.values()])
        rows.append({"trials_rule": k[0], "early_lick": k[1], "H": k[2], "w_trials": which, "sessions": len(x),
                     "mean": x.mean(), "sem": x.std(ddof=1) / np.sqrt(len(x)), "within": abs(x.mean() - 0.61) <= 0.05})
cs = pd.DataFrame(rows)
print(cs.round(3).to_string(index=False))
print("\nw_trials: all = every correct control trial, train = the training subset that builds cd; neither is fixed by the claims file.")
for which in ("all", "train"):
    print(f"Within 0.56 to 0.66 under some H reading, T1, w from {which}, per K reading:", by_k(cs[cs.w_trials == which], "within"))
print("Per session, w from all correct control trials against the training subset (T1, K1, H1):")
v = A[("per condition", "drop", "H1")]
print(pd.DataFrame([{"sid": s, "neurons": r["n_neurons"], "w_corr_all": r["w_corr_all"], "w_corr_train": r["w_corr_train"]} for s, r in v.items()]).round(3).to_string(index=False))

**Result: blocked (methods_underspecified).** With w from all correct control trials the mean
correlation between late sample and late delay w is 0.563 (lick-early trials dropped) and 0.588
(kept) over 24 sessions, within 0.56 to 0.66. With w from the training subset that builds cd it is
0.534 (H1) or 0.554 (H2) dropped and 0.537 or 0.553 kept, below 0.56. The claims file does not say
which trials give w, and the two readings give different outcomes, so the claim is blocked (the
first round gave reproduced, from the first reading alone). Counting trials per trial type (T2, a
variant) gives 0.559 (dropped) and 0.580 (kept) with all trials. The sessions follow the rule as
written (T1, 24 sessions with 6 to 37 neurons, mean 17, against the paper's 16 with 6 to 20, mean
11).

## fig3c-cd-recovery-roc

> Along cd, trajectories recovered after ipsilateral photoinhibition (ROC 0.73 against 0.76 in
> control) and were little changed by contralateral photoinhibition (0.74), but were permanently
> altered by bilateral photoinhibition (0.58).

Criterion: convention A; ROC of cd'x between lick right and lick left trials in the last 400 ms of
the delay, per session, mean over sessions. Reproduced when all four are within the paper's
s.e.m. under H1 or H2; partially when control, ipsilateral and bilateral are and their order holds.

In [ ]:
EXP3C = {"control": (0.76, 0.03), "left1": (0.73, 0.02), "right1": (0.74, 0.03), "bilateral8": (0.58, 0.03)}
rows = []
for k, v in A.items():
    roc = pd.DataFrame([r["roc_go"] for r in v.values()])
    row = {"trials_rule": k[0], "early_lick": k[1], "H": k[2], "sessions": len(roc)}
    for c in CONDS_A:
        row[c] = roc[c].mean(); row[c + "_sem"] = roc[c].sem()
        row[c + "_ok"] = abs(roc[c].mean() - EXP3C[c][0]) <= EXP3C[c][1]
    row["all_four"] = all(row[c + "_ok"] for c in CONDS_A)
    row["partial"] = row["control_ok"] and row["left1_ok"] and row["bilateral8_ok"] and row["control"] >= row["left1"] > row["bilateral8"]
    rows.append(row)
r3c = pd.DataFrame(rows)
print(r3c.round(3).to_string(index=False))
print("\nT1, per K reading: all four within", by_k(r3c, "all_four"), "; partial rule", by_k(r3c, "partial"))
# variant: spread over 10 random splits for one reading
k0 = ("per condition", "drop", "H1")
sp = []
for seed in range(10):
    vals = []
    for sid, uids in L.a_sessions(U, S, "exp2", CONDS_A, "drop", False).items():
        X, t = L.session_counts(U, S, sid, uids, "drop")
        r = L.activity_session(X, t, 0.5, CONDS_A, "left1", np.random.default_rng(1000 + seed + int(sid.replace("_", "")) % 2 ** 20))
        vals.append(r["roc_go"])
    sp.append(pd.DataFrame(vals).mean())
sp = pd.DataFrame(sp)
print("\nVariant, 10 random splits (per condition, lick-early dropped, H1): mean ROC per split, range")
print(sp.agg(["min", "max"]).round(3).to_string())

**Result: not reproduced.** The bilateral value is within tolerance under every reading (0.558 to
0.559 with trials per condition), but the control ROC is 0.649 to 0.677, below the paper's 0.76 by
more than its s.e.m., and the ipsilateral (0.631 to 0.652) and contralateral (0.672 to 0.704) values
are also below theirs. The order holds: ipsilateral and contralateral are close to control, and
bilateral is lowest. Over 10 random training splits the control mean ranges 0.641 to 0.686. The
analysis uses 24 sessions with 6 to 37 neurons, where the paper used 16 with 6 to 20; the answer
confirms the rule as written (T1) is the one to follow. The outcome is the same under K1 and K2.

## ed8-roc-during-photoinhibition

> During photoinhibition trajectory separation along cd fell to 0.54 (ipsilateral), 0.64
> (contralateral) and 0.54 (bilateral), against 0.72 in control.

Criterion: convention A with cd'x in a window during photoinhibition: (a) the last 400 ms before the
ramp, (b) the 400 ms window centred on the photostimulus, (c) the mean over the photostimulus. With
0.8 s of light from delay onset and a 0.2 s ramp, (a) and (b) are the same window (0.2 to 0.6 s
after delay onset). Reproduced when all four are within tolerance under a reading.

In [ ]:
EXP8 = {"control": (0.72, 0.02), "left1": (0.54, 0.03), "right1": (0.64, 0.03), "bilateral8": (0.54, 0.01)}
rows = []
for k, v in A.items():
    for wname, key in (("a/b", "roc_light_ab"), ("c", "roc_light_c")):
        roc = pd.DataFrame([r[key] for r in v.values()])
        row = {"trials_rule": k[0], "early_lick": k[1], "H": k[2], "window": wname}
        for c in CONDS_A:
            row[c] = roc[c].mean(); row[c + "_ok"] = abs(roc[c].mean() - EXP8[c][0]) <= EXP8[c][1]
        row["all_four"] = all(row[c + "_ok"] for c in CONDS_A)
        rows.append(row)
r8 = pd.DataFrame(rows)
print(r8.round(3).to_string(index=False))
r8["n_within"] = r8[[c + "_ok" for c in CONDS_A]].sum(axis=1)
r8["all_below_control"] = (r8[["left1", "right1", "bilateral8"]].lt(r8.control, axis=0)).all(axis=1)
print("\nT1, per K reading: all four within", by_k(r8, "all_four"), "; most values within under a reading",
      r8[r8.trials_rule == "per condition"].groupby("early_lick").n_within.max().to_dict(),
      "; every perturbed value below control under every reading", r8[r8.trials_rule == "per condition"].groupby("early_lick").all_below_control.all().to_dict())


**Result: partially reproduced.** During the light, separation along cd falls below control in every
condition: control 0.649 to 0.685, ipsilateral 0.49 to 0.58, contralateral 0.59 to 0.64, bilateral
0.48 to 0.55 (trials per condition). Under reading (c) with lick-early trials dropped and H1, the
ipsilateral (0.546), contralateral (0.626) and bilateral (0.535) values are within tolerance, but the
control value is below 0.70 under every reading, so no reading has all four. With lick-early trials
kept at most two are within (ipsilateral 0.534 and contralateral 0.631, window a/b, H1). Under K1
and K2 every perturbed value is below control under every reading, so the effect holds in part
under both. The criterion
defines no partial outcome; at least half of the four values hold under K1 (three) and K2 (two), so
the outcome is partial by the rule given in the Summary.

## cd-captures-selectivity

> Projection on cd captured 65.6% of population selectivity and 36.4% of task-related activity
> variance over the sample and delay epochs; modes 1 to 5 together captured 98.5% of variance and
> 95.8% of selectivity.

Criterion: selectivity is the RMS of the trial-averaged lick right minus lick left rate over neurons
and time in the sample and delay epochs; variance the RMS of the baseline-subtracted
trial-averaged activity; captured fraction (a) RMS ratio or (b) its square; baseline (a) the
pre-sample period (the 400 ms window before sample onset), (b) the mean over the trial. Reproduced
when the cd values are within tolerance under a reading; the mode 1-5 values must hold for a full
reproduction, else partial. Answered 2026-10-21 as not given by the paper: which trials give the
trial-averaged activity, (v1) the correct control trials of the training set, from which cd and the
modes are built, or (v2) all correct control trials; the outcome must hold under both, and under
K1 and K2. Sessions follow convention A's rule (T1).

In [ ]:
rows = []
for k, v in A.items():
    for base in ("pre-sample", "trial mean"):
        if base == "pre-sample":
            res = [r for r in v.values() if r["captured"] is not None]
        else:
            res = []
            for sid in v:
                uids = L.a_sessions(U, S, "exp2", CONDS_A, k[1], k[0] == "per type")[sid]
                X, t = L.session_counts(U, S, sid, uids, k[1])
                rng = np.random.default_rng(int(sid.replace("_", "")) % 2 ** 32)
                r_ = L.activity_session(X, t, 0.5 if k[2] == "H1" else 1 / 3, CONDS_A, "left1", rng, baseline="trial mean")
                if r_["captured"] is not None:
                    res.append(r_)
        for vname, key in (("v1 training", "captured"), ("v2 all correct", "captured_v2")):
            cap = pd.DataFrame([r[key] for r in res])
            for ratio in ("RMS", "squared"):
                pw = 1 if ratio == "RMS" else 2
                m = (cap ** pw).mean() * 100
                rows.append({"trials_rule": k[0], "early_lick": k[1], "H": k[2], "trials": vname, "baseline": base, "ratio": ratio,
                             "sessions": len(cap), "sel_cd": m.sel_cd, "var_cd": m.var_cd, "sel_15": m.sel_15, "var_15": m.var_15})
cap = pd.DataFrame(rows)
cap["cd_ok"] = ((cap.sel_cd - 65.6).abs() <= 5.1) & ((cap.var_cd - 36.4).abs() <= 6.3)
cap["modes_ok"] = ((cap.var_15 - 98.5).abs() <= 0.5) & ((cap.sel_15 - 95.8).abs() <= 1.2)
cap["full"] = cap.cd_ok & cap.modes_ok
print(cap.round(1).to_string(index=False))
t1 = cap[cap.trials_rule == "per condition"]
print("\nT1, per K and v reading: cd values within under some reading, and with the mode values as well:")
print(t1.groupby(["early_lick", "trials"]).agg(cd_ok=("cd_ok", "any"), full=("full", "any"),
      sel_cd=("sel_cd", lambda x: f"{x.min():.1f}-{x.max():.1f}"), var_cd=("var_cd", lambda x: f"{x.min():.1f}-{x.max():.1f}"),
      sel_15=("sel_15", lambda x: f"{x.min():.1f}-{x.max():.1f}"), var_15=("var_15", lambda x: f"{x.min():.1f}-{x.max():.1f}")).to_string())


**Result: blocked (methods_underspecified).** The answer gives no rule for which trials' average is
decomposed, and the claims file now requires the outcome under both readings; they differ. Under v1
(the training trials, the reading run before the answer) cd captures 73.7% to 88.0% of the
selectivity with trials per condition, outside 65.6 +/- 5.1 under every reading, so the claim is not
reproduced. Under v2 (all correct control trials), with H2, squared ratios and the pre-sample
baseline, cd captures 66.2% of the selectivity and 34.6% of the variance with lick-early trials
dropped (67.6% and 39.5% kept), both within tolerance, while modes 1 to 5 capture 88.6% to 96.1% of
the selectivity and 91.5% to 97.6% of the variance, outside 95.8 +/- 1.2 and 98.5 +/- 0.5, so v2 is
partial. Under H1, v2 gives 74.4% or more of the selectivity. The captured fractions are measured
on trial averages that include the trials used to build cd and the modes (all of them under v1,
part of them under v2), so they are in-sample; the claims file names no held-out reading, and
none is run.

## fig3c-cd-predicts-movement and fig4-bilateral-cd-predicts-movement

> Activity along cd predicts the upcoming movement trial by trial; deviations toward the lick right
> trajectory predict more lick right responses, and vice versa. (fig3c)

> Even after bilateral photoinhibition destroys average selectivity, single-trial activity along cd
> predicts the upcoming movement. (fig4)

Criterion: for each session the decision boundary on the training control trials; for each held-out
control and perturbation trial the distance of cd'x from the boundary in the last 400 ms before the
go cue; trials pooled over sessions. Reproduced, for control and ipsilateral trials (fig3c) or for
bilateral trials (fig4), when the fraction of trials whose lick falls on the side of the boundary is
above 0.5 by a one-sided binomial test at p < 0.05, and lick right performance rises and lick left
performance falls with distance (Spearman correlation over bins of 4 of the right sign). Bins hold
at least one trial; bins with at least 5 trials are shown as a variant.

In [ ]:
def predicts(trials, cond, min_bin=1):
    d = trials[trials.cond == cond]
    pred = (d.d > 0) == d.licked_right
    k, n = int(pred.sum()), len(d)
    pb = stats.binomtest(k, n, 0.5, alternative="greater").pvalue
    d = d.assign(bin=np.floor(d.d / 4) * 4 + 2)
    out = {"n": n, "frac_pred": k / n, "binom_p": pb}
    for side, name in ((True, "right"), (False, "left")):
        g = d[d.right == side].groupby("bin").hit.agg(["mean", "size"])
        g = g[g["size"] >= min_bin]
        rho = stats.spearmanr(g.index, g["mean"]) if len(g) > 2 else None
        out[f"rho_{name}"] = rho.statistic if rho else np.nan; out[f"p_{name}"] = rho.pvalue if rho else np.nan
        out[f"bins_{name}"] = len(g)
    out["holds"] = (pb < 0.05) and out["rho_right"] > 0 and out["rho_left"] < 0
    return out
rows = []
for k, v in A.items():
    trials = pd.concat([r["trials"] for r in v.values()], ignore_index=True)
    for c in CONDS_A:
        for mb in (1, 5):
            rows.append({"trials_rule": k[0], "early_lick": k[1], "H": k[2], "cond": c, "min_trials_per_bin": mb, **predicts(trials, c, mb)})
pm = pd.DataFrame(rows)
print(pm.round(4).to_string(index=False))
print("\nT1, minimum 1 trial per bin, per K reading: control and ipsilateral hold (fig3c)",
      by_k(pm[(pm.min_trials_per_bin == 1)].groupby(["trials_rule", "early_lick", "H"]).apply(
          lambda d: d[d.cond.isin(["control", "left1"])].holds.all()).rename("h").reset_index(), "h"),
      "; bilateral holds (fig4)", by_k(pm[(pm.min_trials_per_bin == 1) & (pm.cond == "bilateral8")], "holds"))


**Result: fig3c-cd-predicts-movement reproduced; fig4-bilateral-cd-predicts-movement reproduced.**
The side of the decision boundary predicts the lick in 67% to 69% of held-out control trials (1890
to 2895 trials) and 63% to 68% of ipsilateral trials (261 to 448), and in 59% to 64% of bilateral
trials (777 to 1139); every binomial p is below 1e-6. Lick right performance rises with the
distance from the boundary (Spearman rho +0.28 to +0.99 for control and ipsilateral trials, +0.84
to +1.00 for bilateral ones), and lick left performance falls (rho -0.76 to -1.00), under every
reading, with bins of at least one or at least 5 trials. Both hold under K1 and K2 with the rule
as written (T1), as the answers require.

## fig3d-persistent-mode and fig3e-ramping-mode

> A persistent mode, built to capture the sustained effect of ipsilateral perturbation, stays
> altered past movement onset but neither discriminates trial type nor predicts behaviour on control
> trials. (fig3d)

> A ramping mode capturing most remaining variance shows non-selective ramping in the delay epoch,
> does not predict behaviour, and is resistant to unilateral and bilateral perturbation. (fig3e)

Criteria: mode 2 is the difference of mean response vectors, ipsilateral perturbed minus control
lick right correct trials, at the go cue; mode 4 the first SVD eigenvector of the baseline-subtracted
PSTH matrix (sample and delay epochs) made orthogonal to modes 1 to 3. fig3d is reproduced when the
session mean ROC between lick right and lick left control trials is within 2 bootstrap s.e.m. of 0.5,
the fraction predicted is not above 0.5 by the one-sided binomial test, and the ipsilateral minus
control difference at the go cue is outside 2 s.e.m. of zero. fig3e is reproduced when the control
projection rises over the delay (positive in the session mean), its ROC and fraction predicted fail
the tests of fig3d, and the perturbed minus control difference at the go cue is within 2 bootstrap
s.e.m. of zero for ipsilateral, contralateral and bilateral trials. Answered 2026-10-21:
the ipsilateral trials that build modes 2 and 3 are split like the control trials, so projected
ipsilateral trials are ones not used to build them, as done here. The sign of mode 4 is not given;
readings (g1) the mode oriented to a positive weight sum, ramping judged by a positive change over
the delay; (g2) each session's mode oriented so that the control projection at the go cue exceeds
that at delay onset, ramping judged by the size of the change against 2 bootstrap s.e.m., with ROC
and the go-cue differences following the orientation. The outcome must hold under both, and under
K1 and K2 (T1). The bootstrap is over sessions (1000 resamples).


In [ ]:
def boot_sem(x, n=1000, seed=0):
    rng = np.random.default_rng(seed); x = np.asarray(x, float)
    return np.std([rng.choice(x, len(x)).mean() for _ in range(n)])
rows = []
for k, v0 in A.items():
    v = {sid: r for sid, r in v0.items() if r["m2"] is not None}
    for mode in ("m2", "m4"):
        roc = np.array([r[mode]["roc"] for r in v.values()])
        k_ok = sum(r[mode]["n_correct_pred"] for r in v.values()); n_all = sum(r[mode]["n_pred"] for r in v.values())
        row = {"trials_rule": k[0], "early_lick": k[1], "H": k[2], "mode": mode, "sessions": len(v), "dropped": len(v0) - len(v),
               "roc": roc.mean(), "roc_2sem": 2 * boot_sem(roc),
               "frac_pred": k_ok / n_all, "binom_p": stats.binomtest(k_ok, n_all, 0.5, alternative="greater").pvalue}
        row["roc_near_half"] = abs(row["roc"] - 0.5) <= row["roc_2sem"]
        for c in ("left1", "right1", "bilateral8"):
            dg = np.array([r[mode]["diff_go"][c] for r in v.values()])
            row[f"diff_{c}"] = dg.mean(); row[f"diff_{c}_2sem"] = 2 * boot_sem(dg)
        rise = np.array([r[mode]["rise_delay"] for r in v.values()])
        row["rise_delay"] = rise.mean(); row["rise_2sem"] = 2 * boot_sem(rise)
        rows.append(row)
md = pd.DataFrame(rows)
print(md.round(3).to_string(index=False))
m2 = md[md["mode"] == "m2"].copy()
m2["holds_3d"] = m2.roc_near_half & (m2.binom_p >= 0.05) & (m2.diff_left1.abs() > m2.diff_left1_2sem)
m4 = md[md["mode"] == "m4"].copy()
m4["holds_3e"] = (m4.rise_delay > 0) & m4.roc_near_half & (m4.binom_p >= 0.05) & \
    np.all([m4[f"diff_{c}"].abs() <= m4[f"diff_{c}_2sem"] for c in ("left1", "right1", "bilateral8")], axis=0)
print("\nfig3d:"); print(m2[["trials_rule", "early_lick", "H", "roc", "roc_2sem", "frac_pred", "binom_p", "diff_left1", "diff_left1_2sem", "holds_3d"]].round(3).to_string(index=False))
print("\nfig3e:"); print(m4[["trials_rule", "early_lick", "H", "rise_delay", "rise_2sem", "roc", "roc_2sem", "frac_pred", "binom_p",
                           "diff_left1", "diff_left1_2sem", "diff_right1", "diff_right1_2sem", "diff_bilateral8", "diff_bilateral8_2sem", "holds_3e"]].round(3).to_string(index=False))
# Reading g2 for mode 4: per session, the mode oriented so that the control projection rises from delay onset to the go
# cue; ramping judged by the size of that change against 2 bootstrap s.e.m. ROC, the go-cue differences and the
# predictions follow the orientation (ROC becomes 1 - ROC and differences change sign where the mode is flipped; the
# fraction predicted does not depend on the sign).
rows = []
for k, v0 in A.items():
    v = {sid: r for sid, r in v0.items() if r["m4"] is not None}
    sgn = np.array([1.0 if r["m4"]["rise_delay"] >= 0 else -1.0 for r in v.values()])
    rise = np.array([abs(r["m4"]["rise_delay"]) for r in v.values()])
    roc = np.array([r["m4"]["roc"] if s > 0 else 1 - r["m4"]["roc"] for s, r in zip(sgn, v.values())])
    row = {"trials_rule": k[0], "early_lick": k[1], "H": k[2], "sessions": len(v), "flipped": int((sgn < 0).sum()),
           "rise_delay": rise.mean(), "rise_2sem": 2 * boot_sem(rise), "roc": roc.mean(), "roc_2sem": 2 * boot_sem(roc)}
    for c in ("left1", "right1", "bilateral8"):
        dg = sgn * np.array([r["m4"]["diff_go"][c] for r in v.values()])
        row[f"diff_{c}"] = dg.mean(); row[f"diff_{c}_2sem"] = 2 * boot_sem(dg)
    row["frac_pred"] = m4.set_index(["trials_rule", "early_lick", "H"]).loc[k, "frac_pred"]
    row["binom_p"] = m4.set_index(["trials_rule", "early_lick", "H"]).loc[k, "binom_p"]
    rows.append(row)
m4g2 = pd.DataFrame(rows)
m4g2["ramps"] = m4g2.rise_delay > m4g2.rise_2sem
m4g2["roc_near_half"] = (m4g2.roc - 0.5).abs() <= m4g2.roc_2sem
m4g2["holds_3e"] = m4g2.ramps & m4g2.roc_near_half & (m4g2.binom_p >= 0.05) & \
    np.all([m4g2[f"diff_{c}"].abs() <= m4g2[f"diff_{c}_2sem"] for c in ("left1", "right1", "bilateral8")], axis=0)
print("\nfig3e, reading g2:"); print(m4g2.round(3).to_string(index=False))
print("\nT1, per K reading: fig3d holds", by_k(m2, "holds_3d"), "; fig3e holds under g1", by_k(m4, "holds_3e"), "and under g2", by_k(m4g2, "holds_3e"))
print("fig3e parts, T1: predicts above chance (binomial p < 0.05) under every reading",
      bool((m4[m4.trials_rule == "per condition"].binom_p < 0.05).all()), "; g1 rise positive", by_k(m4.assign(r=m4.rise_delay > 0), "r"),
      "; g2 ramps beyond 2 s.e.m.", by_k(m4g2, "ramps"))


# Per-session ROC of modes 2 and 4 on held-out control trials (verifier's request), T1, with |ROC - 0.5|.
# Mode 4 is shown as oriented by g1 (positive weight sum); under g2 a flipped session has 1 - ROC, and
# |ROC - 0.5| is the same.
ps = []
for k, v0 in A.items():
    if k[0] != "per condition":
        continue
    for sid, r in v0.items():
        if r["m2"] is None:
            continue
        ps.append({"early_lick": k[1], "H": k[2], "sid": sid, "neurons": r["n_neurons"], "roc_m2": r["m2"]["roc"], "roc_m4": r["m4"]["roc"]})
ps = pd.DataFrame(ps)
ps["abs_m2"] = (ps.roc_m2 - 0.5).abs(); ps["abs_m4"] = (ps.roc_m4 - 0.5).abs()
print(ps.pivot(index="sid", columns=["early_lick", "H"], values=["roc_m2", "roc_m4"]).round(3).to_string())
print("\nMean |ROC - 0.5| over sessions, with 2 bootstrap s.e.m.; for reference the cd ROC at the go cue on the same sessions:")
for (e_, h_), d in ps.groupby(["early_lick", "H"]):
    cd_roc = np.array([r["roc_go"]["control"] for s_, r in A[("per condition", e_, h_)].items() if s_ in set(d.sid)])
    print(f"  {e_}, {h_}: mode 2 {d.abs_m2.mean():.3f} (2 s.e.m. {2 * boot_sem(d.abs_m2):.3f}); mode 4 {d.abs_m4.mean():.3f} "
          f"(2 s.e.m. {2 * boot_sem(d.abs_m4):.3f}); cd |ROC - 0.5| {np.mean(np.abs(cd_roc - 0.5)):.3f}; sessions {len(d)}")

**Result: fig3d-persistent-mode not reproduced; fig3e-ramping-mode not reproduced.** The answer
confirmed the split of the ipsilateral trials used here, and both outcomes are the same under K1 and
K2, and for fig3e under g1 and g2. The persistent mode is altered by ipsilateral photoinhibition up
to the go cue (difference 1.08 to 1.75, more than its 2 s.e.m. of 0.57 to 0.66), but it predicts the
lick on control trials above chance (59% to 62% of trials, binomial p below 1e-6) under every
reading, and under two of four readings with trials per condition its ROC (0.408, 0.430) is further
than 2 s.e.m. from 0.5. The ramping mode is resistant to all three perturbations under the rule as written (T1): the
differences are within 2 s.e.m. under every reading and either orientation. Under the T2 variant
with lick-early trials dropped, H1 and g1, the ipsilateral difference (-0.353) is beyond its 2
s.e.m. (0.294). Under g1 its change over the delay is
-0.73 to +0.11, positive under one of four readings, and its ROC is near 0.5 under three of four;
under g2 the change is 1.26 to 1.56 against 2 s.e.m. of 0.47 to 0.53, so it ramps, and its ROC is
near 0.5 under all four. Under both it predicts the lick in 55% to 58% of control trials, above
chance (p below 1e-6), so it fails the claim.

Per session (T1), the mean of |ROC - 0.5| on held-out control trials is 0.091 to 0.127 for the
persistent mode and 0.064 to 0.103 for the ramping mode, against 0.154 to 0.181 for cd on the same
sessions. A session-mean ROC near 0.5 therefore partly reflects separation of opposite sign in
different sessions; the binomial test of the fraction predicted, which does not depend on the sign,
fails both claims under every reading.

## ed6b-session-bias-predicted

> In each session the behavioural bias after bilateral photoinhibition followed the side of the
> control trajectory that the perturbed cd trajectory ended closer to.

Criterion: per session with bilateral photoinhibition, the mean cd'x of bilateral trials at the go
cue relative to the decision boundary (convention A), and the behavioural bias of
ed6a-bilateral-inconsistent-bias; reproduced when the signs agree in a majority of sessions, above
chance by a one-sided binomial test at p < 0.05.

In [ ]:
rows = []
for k, v in A.items():
    agree = []
    for sid, r in v.items():
        tb = r["trials"]; side = tb[tb.cond == "bilateral8"].d.mean()
        agree.append({"sid": sid, "side": side, "bias": BIAS.loc[sid, "bias"], "agree": np.sign(side) == np.sign(BIAS.loc[sid, "bias"])})
    ag = pd.DataFrame(agree)
    kk = int(ag.agree.sum())
    rows.append({"trials_rule": k[0], "early_lick": k[1], "H": k[2], "sessions": len(ag), "agree": kk,
                 "binom_p": stats.binomtest(kk, len(ag), 0.5, alternative="greater").pvalue})
b6b = pd.DataFrame(rows)
b6b["holds"] = (b6b.agree > b6b.sessions / 2) & (b6b.binom_p < 0.05)
print(b6b.round(4).to_string(index=False))
print("T1, per K reading:", by_k(b6b, "holds"))
print("\nPer session, one reading (per condition, lick-early dropped, H1):")
print(pd.DataFrame([{"sid": s, "side": r["trials"][r["trials"].cond == "bilateral8"].d.mean(),
      "bias": BIAS.loc[s, "bias"]} for s, r in A[("per condition", "drop", "H1")].items()]).round(3).to_string(index=False))

**Result: not reproduced.** The side of the boundary at which the bilateral trials end agrees in sign
with the behavioural bias in 11 to 15 of 19 to 24 sessions, not above chance (one-sided binomial p
0.15 to 0.50) under any reading. The legend's five example sessions are not identified here. With the rule as written (T1) it is
13 to 15 of 24 sessions (p 0.15 to 0.42), not above chance under K1 or K2, so the answers leave the
outcome unchanged.

## ed9f-reaction-time

> Reaction times were faster on trials whose cd trajectory was far from the decision boundary at
> the go cue.

Criterion: convention A; reaction time is the first lick after the go cue, relative to the session
mean; distance is the absolute distance of cd'x from the boundary at the go cue; lick right and lick
left trials pooled; readings (a) control trials, (b) all trials. Reproduced when the Spearman
correlation between distance and reaction time is negative with p < 0.05. Answered 2026-10-21: the
paper does not say how a lick is detected; the files hold no lick times, only `lick_trace`, an
analogue signal ("the animal's tongue movement when licking") in 5 s chunks at 10 kHz. The outcome
stands only if it is the same over the thresholds tried, else `methods_underspecified`. The
thresholds tried here, all choices over a range: the raw signal rising above 0.10, 0.12 or 0.14
(level), above the pre-go baseline mean plus 0.01, 0.02 or 0.05 (rise), and above the baseline mean
plus 3, 5, 10 or 20 baseline standard deviations (sd). The baseline is the 300 ms before the go cue,
and the lick is searched up to 1.4 s after it (the chunk ends 1.43 s after the go cue in most
trials). Projected trials are the held-out trials of convention A; a trial without a detected lick
under a rule leaves that rule's test.

In [ ]:
# lick_trace windows around the go cue for every trial of the convention A sessions
t0 = time.time()
RULES = {"level 0.10": ("level", 0.10), "level 0.12": ("level", 0.12), "level 0.14": ("level", 0.14),
         "level 0.16": ("level", 0.16), "level 0.20": ("level", 0.20), "level 0.25": ("level", 0.25),
         "rise 0.01": ("rise", 0.01), "rise 0.02": ("rise", 0.02), "rise 0.05": ("rise", 0.05),
         "sd 3": ("sd", 3), "sd 5": ("sd", 5), "sd 10": ("sd", 10), "sd 20": ("sd", 20)}
LK, lk_rows = {}, []
for sid in sorted(set().union(*[set(v) for v in A.values()])):
    s = S[sid]; tr = s["trials"]
    aid = assets[assets.path == s["path"]].asset_id.iloc[0]
    df, info = L.lick_onsets(aid, tr.go.values, RULES)
    df["go"] = tr.go.values; df["outcome"] = tr.outcome.values; df["early_lick"] = tr.early_lick.values
    LK[sid] = df
    h = df[df.read & (df.outcome == "hit") & ~df.early_lick]
    lk_rows.append({"sid": sid, "chunks": info["chunks"], "nonuniform_chunks": info["chunks_nonuniform"],
                    "behaving": int(tr.behaving.sum()), "read": int((df.read & tr.behaving.values).sum()),
                    "baseline": h.baseline.median(), "max_after_go": h.max_after.median(),
                    **{f"hit_detected_{r}": (h[r].notna().mean() if len(h) else np.nan) for r in ("level 0.12", "rise 0.02", "sd 5", "sd 20")},
                    "median_rt_ms_sd5": h["sd 5"].median() * 1000})
lk = pd.DataFrame(lk_rows)
print(f"lick_trace windows read for {len(lk)} sessions in {time.time() - t0:.0f} s")
print(lk.round(3).to_string(index=False))
print("\nSessions where a lick is detected (sd 5) on more than half of correct trials without an early lick:",
      int((lk["hit_detected_sd 5"] > 0.5).sum()), "of", len(lk))

# Does the trace carry licks? Detection on correct trials (the mouse licked) against no-response trials (it did not),
# pooled over the sessions, and the same for the first session (by id) of each other group of the dandiset.
def lick_contrast(df):
    ok = df.read & ~df.early_lick
    h, n = df[ok & (df.outcome == "hit")], df[ok & (df.outcome == "ignore")]
    return {"hit_trials": len(h), "no_response_trials": len(n), "max_minus_baseline_hit_median": (h.max_after - h.baseline).median(),
            **{f"{r}": f"{h[r].notna().mean():.2f}/{n[r].notna().mean() if len(n) else np.nan:.2f}" for r in RULES}}
print("\nFraction of trials with a detected lick, correct / no-response (trials without an early lick):")
print("convention A sessions:", lick_contrast(pd.concat(LK.values(), ignore_index=True)))
ref = []
for exp in ("exp1", "exp3", "no_photostim_record"):
    sid = sorted(k for k, s_ in S.items() if s_["exp"] == exp and s_["trials"].behaving.any())[0]
    tr = S[sid]["trials"]
    df, _ = L.lick_onsets(assets[assets.path == S[sid]["path"]].asset_id.iloc[0], tr.go.values, RULES)
    df["outcome"] = tr.outcome.values; df["early_lick"] = tr.early_lick.values
    ref.append({"group": exp, "sid": sid, **lick_contrast(df)})
print(pd.DataFrame(ref).round(4).to_string(index=False))


In [ ]:
rows = []
for k, v in A.items():
    tt = []
    for sid, r in v.items():
        tb = r["trials"].merge(LK[sid].drop(columns=["outcome", "early_lick"]), on="go", how="left")
        tb["sid"] = sid
        tt.append(tb)
    tt = pd.concat(tt, ignore_index=True)
    for reading, sel in (("a control", tt.cond == "control"), ("b all", tt.cond.notna())):
        for rule in RULES:
            d = tt[sel & tt[rule].notna()].copy()
            d["rt_rel"] = d[rule] - d.groupby("sid")[rule].transform("mean")
            rho = stats.spearmanr(d.d.abs(), d.rt_rel) if len(d) > 2 else None
            rows.append({"trials_rule": k[0], "early_lick": k[1], "H": k[2], "trials": reading, "rule": rule,
                         "n": len(d), "sessions": d.sid.nunique(), "median_rt_ms": d[rule].median() * 1000,
                         "rho": rho.statistic if rho else np.nan, "p": rho.pvalue if rho else np.nan})
rt = pd.DataFrame(rows)
rt["holds"] = (rt.rho < 0) & (rt.p < 0.05)
print(rt[rt.trials_rule == "per condition"].round(4).to_string(index=False))
j9 = rt[rt.trials_rule == "per condition"].groupby(["early_lick", "rule"]).holds.any().unstack(0)
print("\nT1: holds under some reading (a or b, H1 or H2), per threshold and K reading:")
print(j9.to_string())
print("Thresholds and K readings under which it holds:", int(j9.values.sum()), "of", j9.size)
print("T2 (variant):", rt[rt.trials_rule == "per type"].groupby("early_lick").holds.mean().round(2).to_dict(), "fraction of rows holding")
print("rho range, T1:", rt[rt.trials_rule == "per condition"].rho.min().round(3), "to", rt[rt.trials_rule == "per condition"].rho.max().round(3))


In [ ]:
# Raw lick_trace (verifier's request): per session, the pre-go baseline and the rise after the go cue
# on correct and on no-response trials, for every convention A session (experiment 2) and the
# reference sessions; then example windows of four correct trials from two experiment-2 and two
# experiment-3 sessions.
def trace_summary(df, sid, group):
    ok = df.read & ~df.early_lick
    h, n = df[ok & (df.outcome == "hit")], df[ok & (df.outcome == "ignore")]
    r = {"group": group, "sid": sid, "hit": len(h), "no_response": len(n), "baseline_median": h.baseline.median(),
         "baseline_sd_median": h.baseline_sd.median()}
    for name, d in (("hit", h), ("noresp", n)):
        rise = d.max_after - d.baseline
        r[f"rise_{name}_q50_q90_q99"] = "/".join(f"{x:.4f}" for x in rise.quantile([0.5, 0.9, 0.99])) if len(d) else ""
    return r
ex3 = sorted(k for k, s_ in S.items() if s_["exp"] == "exp3" and s_["trials"].behaving.any())
LK3 = {}
for sid in ex3:
    tr = S[sid]["trials"]
    df, _ = L.lick_onsets(assets[assets.path == S[sid]["path"]].asset_id.iloc[0], tr.go.values, RULES)
    df["go"] = tr.go.values; df["outcome"] = tr.outcome.values; df["early_lick"] = tr.early_lick.values
    LK3[sid] = df
summ_rows = [trace_summary(df, sid, "exp2 (convention A)") for sid, df in LK.items()] + [trace_summary(df, sid, "exp3") for sid, df in LK3.items()]
print(pd.DataFrame(summ_rows).round(4).to_string(index=False))
fig, axes = plt.subplots(1, 4, figsize=(14, 2.6), sharey=False)
tt_ = np.arange(-0.3, 1.4, 1e-4)[:17000]
for ax, sid in zip(axes, sorted(LK)[:2] + ex3[:2]):
    tr = S[sid]["trials"]
    g = tr[(tr.outcome == "hit") & ~tr.early_lick & tr.behaving].go.values[:4]
    ws = L.lick_windows(assets[assets.path == S[sid]["path"]].asset_id.iloc[0], g)
    for k_, w_ in enumerate(ws):
        if w_ is None:
            continue
        ax.plot(tt_[:len(w_)], w_, lw=0.6)
        print(f"{sid} ({S[sid]['exp']}), correct trial {k_}: baseline {w_[:3000].mean():.4f}, min {w_.min():.4f}, max {w_.max():.4f},"
              f" max after go minus baseline {w_[3000:].max() - w_[:3000].mean():.4f}")
    ax.axvline(0, color="k", lw=0.5); ax.set_title(f"{sid} ({S[sid]['exp']})", fontsize=8); ax.set_xlabel("s from go cue")
axes[0].set_ylabel("lick_trace (a.u.)")
plt.tight_layout(); plt.show()
print("Every plotted sample is inside its axis (each panel's y range is set from its data).")

In [ ]:
# Variant (verifier's suggestion; reference only, the claim's sessions are those of experiment 2):
# the same test on experiment 3, whose lick trace carries licks. Convention A on the experiment-3
# sessions (more than 5 neurons with more than 5 trials in each of control, left4, right4 and
# bilateral2), cd and boundary as for ed7e, reaction time from the trace under every rule.
rows = []
for early in ("drop", "keep"):
    sess3 = L.a_sessions(U, S, "exp3", ["control", "left4", "right4", "bilateral2"], early, False)
    for frac, hname in ((0.5, "H1"), (1 / 3, "H2")):
        tt = []
        for sid, uids in sess3.items():
            X, t = L.session_counts(U, S, sid, uids, early)
            r = L.activity_session(X, t, frac, ["control", "left4", "right4", "bilateral2"], "left4", np.random.default_rng(int(sid.replace("_", "")) % 2 ** 32))
            tb = r["trials"].merge(LK3[sid].drop(columns=["outcome", "early_lick"]), on="go", how="left"); tb["sid"] = sid
            tt.append(tb)
        tt = pd.concat(tt, ignore_index=True)
        for reading, sel in (("a control", tt.cond == "control"), ("b all", tt.cond.notna())):
            for rule in RULES:
                d = tt[sel & tt[rule].notna()].copy()
                d["rt_rel"] = d[rule] - d.groupby("sid")[rule].transform("mean")
                rho = stats.spearmanr(d.d.abs(), d.rt_rel) if len(d) > 2 else None
                rows.append({"early_lick": early, "H": hname, "trials": reading, "rule": rule, "sessions": d.sid.nunique(), "n": len(d),
                             "median_rt_ms": d[rule].median() * 1000, "rho": rho.statistic if rho else np.nan, "p": rho.pvalue if rho else np.nan})
rt3 = pd.DataFrame(rows)
rt3["holds"] = (rt3.rho < 0) & (rt3.p < 0.05)
print(rt3.round(4).to_string(index=False))
j3 = rt3.groupby(["early_lick", "rule"]).holds.any().unstack(0)
print("\nExperiment 3: holds under some reading (a or b, H1 or H2), per threshold and K reading:")
print(j3.to_string())
print("Thresholds and K readings under which it holds:", int(j3.values.sum()), "of", j3.size, "; rho range", rt3.rho.min().round(3), "to", rt3.rho.max().round(3))

**Result: blocked (data_missing).** The answer let the claim be tested over a range of lick
thresholds, and the test shows that the lick trace of these sessions holds no licks. In the 24
experiment-2 sessions of convention A the trace sits at a baseline of 0.166 to 0.193 and its median
rise after the go cue on correct trials, where the mouse licked, is 0.0013 to 0.0022 per session
(99th percentile at most 0.0159), the same as on no-response trials (0.0013 to 0.0021); the example
traces show the same. After the first verification round, raw levels of 0.16, 0.20 and 0.25 were
added, since the levels 0.10 to 0.14 lie below the baseline of these sessions. Pooled over 3660
correct and 61 no-response trials, no rule finds a lick on correct trials more often than on
no-response trials (level 0.16, 0.37 against 0.56; 3 s.d., 0.59 against 0.38; level 0.14, 0.05
against 0.03; every other rule 0.02 or less), and no session has a lick detected (5 s.d.) on more
than half of its correct trials. The correlation holds under 5 of the 26 combinations of threshold
and lick-early reading, on 3 to 10 trials, so by the claims file's rule the outcome would also be
`methods_underspecified`. In experiment 3 the trace carries licks: the median rise on correct
trials is 0.0385 to 0.0717 per session, against 0.0017 to 0.0213 on no-response trials. As a
variant on those 6 sessions (a different experiment from the claim's), the correlation is negative
with p < 0.05 under 18 of the 26 combinations; it fails for the levels that lie near or above the
trace (0.10, 0.20, 0.25) and for a rise of 0.05, and its rho ranges from -0.256 to 0.172. The lick
times the claim needs are not in the dandiset for the claim's sessions.

## ed9c-cd-window-robust

> The recovery along cd after unilateral perturbation does not depend on the window used to compute
> cd (first or last 400 ms of the delay epoch).

Criterion: as fig3c-cd-recovery-roc with cd from the first or the last 400 ms of the delay epoch;
reproduced when, for both windows, the ipsilateral ROC at the end of the delay is within 0.05 of the
control ROC and the bilateral ROC is lower than the ipsilateral. A 400 ms window inside a 400 ms
period is a single window, so cd is w of that window.

In [ ]:
rows = []
for per_type in (False, True):
    for early in ("drop", "keep"):
        sess = L.a_sessions(U, S, "exp2", CONDS_A, early, per_type)
        for frac, hname in ((0.5, "H1"), (1 / 3, "H2")):
            for wname, win in (("first 400 ms", (-1.7, -1.3)), ("last 400 ms", (-0.4, 0.0))):
                roc = []
                for sid, uids in sess.items():
                    X, t = L.session_counts(U, S, sid, uids, early)
                    rng = np.random.default_rng(int(sid.replace("_", "")) % 2 ** 32)
                    roc.append(L.activity_session(X, t, frac, CONDS_A, "left1", rng, cd_window=win)["roc_go"])
                roc = pd.DataFrame(roc).mean()
                rows.append({"trials_rule": "per type" if per_type else "per condition", "early_lick": early, "H": hname, "cd_window": wname,
                             **roc.to_dict()})
r9c = pd.DataFrame(rows)
r9c["holds"] = ((r9c.left1 - r9c.control).abs() <= 0.05) & (r9c.bilateral8 < r9c.left1)
print(r9c.round(3).to_string(index=False))
print(r9c.groupby(["trials_rule", "early_lick", "H"]).holds.all().to_string())
hk = r9c.groupby(["trials_rule", "early_lick", "H"]).holds.all().rename("both_windows").reset_index()
print("\nT1, per K reading, both windows hold under some H reading:", by_k(hk, "both_windows"))


**Result: reproduced.** With cd from the first or the last 400 ms of the delay, the ipsilateral ROC is
within 0.05 of the control ROC and the bilateral ROC is below the ipsilateral one under H1 and H2
with lick-early trials dropped, and under H2 with them kept (under H1 kept, the ipsilateral ROC with
the first window is 0.059 to 0.072 below control). The criterion allows either H reading, so the
claim holds under both lick-early readings, as the answer on K requires, with the rule as written
(T1).

## ed7e-roc-larger-photoinhibition

> Trajectory separation along cd at the end of the delay was 0.80 in control, 0.64 after
> ipsilateral and 0.68 after contralateral four-spot photoinhibition, and 0.54 after bilateral
> one-spot photoinhibition.

Criterion: convention A on experiment 3 sessions with more than 5 simultaneously recorded neurons
tested on more than 3 trials per condition (control, ipsilateral four spots, contralateral four
spots, bilateral one spot per side). Tolerances are the legend's s.e.m. (the bilateral 0.8 read as
0.08). Reproduced when all four are within tolerance under H1 or H2.

In [ ]:
CONDS_7 = ["control", "left4", "right4", "bilateral2"]
EXP7E = {"control": (0.80, 0.08), "left4": (0.64, 0.10), "right4": (0.68, 0.15), "bilateral2": (0.54, 0.08)}
rows, A7 = [], {}
for early in ("drop", "keep"):
    sess = L.a_sessions(U, S, "exp3", CONDS_7, early, False, min_trials=3)
    for frac, hname in ((0.5, "H1"), (1 / 3, "H2")):
        res = {}
        for sid, uids in sess.items():
            X, t = L.session_counts(U, S, sid, uids, early)
            res[sid] = L.activity_session(X, t, frac, CONDS_7, "left4", np.random.default_rng(int(sid.replace("_", "")) % 2 ** 32))
        A7[(early, hname)] = res
        roc = pd.DataFrame([r["roc_go"] for r in res.values()])
        row = {"early_lick": early, "H": hname, "sessions": len(roc), "mice": len({s.split('_')[0] for s in res}),
               "neurons": f"{min(r['n_neurons'] for r in res.values())}-{max(r['n_neurons'] for r in res.values())}"}
        for c in CONDS_7:
            row[c] = roc[c].mean(); row[c + "_sem"] = roc[c].sem(); row[c + "_ok"] = abs(roc[c].mean() - EXP7E[c][0]) <= EXP7E[c][1]
        row["all_four"] = all(row[c + "_ok"] for c in CONDS_7)
        rows.append(row)
r7e = pd.DataFrame(rows)
print(r7e.round(3).to_string(index=False))

**Result: partially reproduced.** The answer on lick-early trials requires the outcome under K1
and K2, and it is partial under both. Over 6 sessions of 3 mice (6 to 23 neurons), the ipsilateral
(0.60 to 0.67) and contralateral (0.69 to 0.71) four-spot values are within tolerance, and the
bilateral one-spot value (0.615 to 0.616) is within it with lick-early trials dropped (0.646 to
0.665 kept, outside). The control ROC (0.680 to 0.689) is below the paper's 0.80 by more than its
s.e.m., so no reading has all four; bilateral separation is lower than ipsilateral
only under K1 with H2 (0.616 against 0.672). The criterion defines no partial
outcome; three of the four values hold under K1 and two under K2, so the outcome is partial by the
rule given in the Summary.

## ed7-larger-photoinhibition

> Bilateral photoinhibition with one spot per side changed selectivity more persistently than
> unilateral photoinhibition with four spots; selectivity recovered to 80% of control 490 ms after
> ipsilateral and 235 ms after contralateral four-spot photoinhibition, and did not recover after
> bilateral one-spot photoinhibition; recovery after unilateral photoinhibition was less robust with
> larger photoinhibition.

Criterion: convention D on experiment 3. Reproduced when both times are within tolerance,
bilateral one-spot selectivity stays below 80% of control at the end of the delay, and the
selectivity change at the go cue is larger (more negative) after bilateral one-spot than after
ipsilateral four-spot photoinhibition; "less robust with size" is judged by the ipsilateral
four-spot ROC at the end of the delay (ed7e) lower than the ipsilateral one-spot ROC (fig3c), by
sign. The method details answered 2026-10-21 (K, D, L, E) are run as for fig1g, and the outcome
must be the same under all; trials are counted per condition (N1).

In [ ]:
OFF3 = {"left4": -0.9, "right4": -0.9, "bilateral2": -0.9}
e3 = U[(U.exp == "exp3") & (U.cls == "pyr")]
rec3, courses3 = {}, {}
for early in ("drop", "keep"):
    rec3[early], courses3[early] = L.recovery_readings(e3, S, OFF3, SEL[early], early, no_recovery=("bilateral2",), per_types=(False,))
rec3 = pd.concat(rec3.values(), ignore_index=True).rename(columns={"early": "early_lick"})
dsel = []
for early in ("drop", "keep"):
    for key, (tl, cm) in courses3[early].items():
        jgo = int(np.argmin(abs(tl - 0.0)))
        dsel.append({"early_lick": early, "bin": key[0], "pref": key[1], "S": key[3], "label": key[4],
                     "dsel_go_bilateral2": cm["bilateral2"][jgo] - cm["control"][jgo], "dsel_go_left4": cm["left4"][jgo] - cm["control"][jgo]})
rec3 = rec3.merge(pd.DataFrame(dsel), on=["early_lick", "bin", "pref", "S", "label"])
rec3["ipsi_ok"] = (rec3.left4 - 490).abs() <= 280
rec3["contra_ok"] = (rec3.right4 - 235).abs() <= 156
rec3["n_ok"] = (rec3.n - 40).abs() <= 4
rec3["bil_no_recovery"] = ~rec3.bilateral2_recovers_by_go
rec3["bil_larger_change"] = rec3.dsel_go_bilateral2 < rec3.dsel_go_left4
rec3["holds_D"] = rec3.ipsi_ok & rec3.contra_ok & rec3.bil_no_recovery & rec3.bil_larger_change
print(rec3[["early_lick", "pref", "label", "search", "bin", "S", "n", "left4", "left4_boot", "left4_sd", "right4", "right4_boot", "right4_sd",
            "bilateral2", "bilateral2_max_ratio_to_go", "dsel_go_bilateral2", "dsel_go_left4", "holds_D"]].round(2).to_string(index=False))
roc_cmp = []
for early in ("drop", "keep"):
    for h in ("H1", "H2"):
        r4 = r7e[(r7e.early_lick == early) & (r7e.H == h)].left4.iloc[0]
        r1 = r3c[(r3c.trials_rule == "per condition") & (r3c.early_lick == early) & (r3c.H == h)].left1.iloc[0]
        roc_cmp.append({"early_lick": early, "H": h, "roc_ipsi_4_spots": r4, "roc_ipsi_1_spot": r1, "less_robust": r4 < r1})
roc_cmp = pd.DataFrame(roc_cmp)
print("\n", roc_cmp.round(3).to_string(index=False))
s7 = rec3.groupby(["early_lick", "pref", "label", "search"]).agg(
    sum=("holds_D", "sum"), size=("holds_D", "size"), left4_ms=("left4", lambda x: f"{x.min():.0f}-{x.max():.0f}"),
    right4_ms=("right4", lambda x: f"{x.min():.0f}-{x.max():.0f}"), ipsi_ok=("ipsi_ok", "sum"), contra_ok=("contra_ok", "sum"),
    bil_no_recovery=("bil_no_recovery", "sum"), bil_larger_change=("bil_larger_change", "sum")).reset_index()
s7 = s7.merge(roc_cmp.groupby("early_lick").less_robust.any().reset_index(), on="early_lick")
s7["holds"] = (s7["sum"] > 0) & s7.less_robust
s7["outcome_here"] = np.where(s7.holds, "reproduced", "not")
print("\n", s7.to_string(index=False))
print("Outcomes over the method combinations (K, D, L, E):", s7.outcome_here.value_counts().to_dict())
print("Included neurons:", rec3.n.min(), "to", rec3.n.max(), "; paper 40 (tolerance 4)")

**Result: blocked (methods_underspecified).** The answer gives neither the window label nor the
other details, and the claims file now requires one outcome over all of them (K1, K2; D1 to D3; L1,
L2; E1, E2); they differ. 10 of the 24 combinations have readings that hold, all with centre labels
(L1), and 14 have none, among them every combination with end labels (L2). With lick-early trials
dropped and centre labels the ipsilateral four-spot recovery is 650 to 700 ms (paper 490 +/- 280)
and the contralateral one 50 to 350 ms (235 +/- 156); the selectivity change at the go cue is -0.38
to -0.49 after bilateral one-spot and -0.10 to +0.14 after ipsilateral four-spot photoinhibition.
With end labels the ipsilateral time is 750 ms or more; 750 ms is inside the tolerance (210 to 770
ms), and the ipsilateral time holds in 8 of 16 readings for lick-early trials dropped with D2 or D3
and end labels, but these combinations fail on the other parts. With the preference set
per bin (D3), bilateral selectivity reaches up to 0.87 of control before the go cue in half the
readings. The end of the search (E1, E2) does not change which combinations hold. The four-spot
ipsilateral ROC is below the one-spot ROC under H1 (0.607 against 0.652 with lick-early trials
dropped, 0.604 against 0.631 kept). 58 to 62 neurons are included, against the paper's 40. With lick-early trials kept, 1% to 4% of the
experiment-3 photoinhibition trials have their light before its logged time (data check).

## ed1-monolithic-models-fail

> Monolithic network models (integrator, integrator with corrective feedback, trained random
> recurrent networks) do not recover the unperturbed trajectory after transient silencing of half
> their neurons.

Criterion: simulations with the parameters of the Methods and the Extended Data Fig. 1 legend
(simple integrator N = 100, tau 100 ms, one eigenvalue 0.99; FORCE network N = 400, tau 200 ms,
tanh, p = 0.1, gain 1.5, 30 iterations; silencing 0.2 to 1.0 s of 2 s); 1 ms step, fixed seeds, at
least 10 networks. Reproduced when, at 2 s, the output of each model differs from its unperturbed
output by more than 10% of the unperturbed selectivity, for the simple integrator and the
transient-input FORCE network. The FORCE target (a ramp to plus or minus 1 at 2 s), the feedback
weights (uniform from -1 to 1), the learning rate and the update every 2 ms are standard choices
the paper does not give. The integrator with corrective feedback and the tamed-chaos network lack
parameters in the text and are not built.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
t0 = time.time()
rows = []
for seed in range(10):
    sil = np.random.default_rng(100 + seed).choice(100, 50, replace=False)
    for name, fn in (("integrator, persistent input", lambda **k: L.simple_integrator(**k)),
                     ("integrator, transient input", lambda **k: L.simple_integrator(transient=True, **k)),
                     ("cascaded integrators, transient input", lambda **k: L.cascaded_integrator(**k))):
        _, o = fn(seed=seed); _, p = fn(seed=seed, silence=sil)
        sel = o[1][-1] - o[-1][-1]
        rows.append({"model": name, "seed": seed, "selectivity": sel,
                     "max_rel_diff": max(abs(p[1][-1] - o[1][-1]), abs(p[-1][-1] - o[-1][-1])) / abs(sel)})
with ProcessPoolExecutor(4) as ex:
    force = list(ex.map(L.force_test, range(10)))
for r in force:
    sel = r["unpert_right"] - r["unpert_left"]
    rows.append({"model": "FORCE network, transient input", "seed": r["seed"], "selectivity": sel,
                 "max_rel_diff": max(abs(r["pert_right"] - r["unpert_right"]), abs(r["pert_left"] - r["unpert_left"])) / abs(sel),
                 "train_err": r["train_err"]})
mono = pd.DataFrame(rows)
print(f"simulated in {time.time() - t0:.0f} s")
print(mono.round(3).to_string(index=False))
ms = mono.groupby("model").agg(networks=("seed", "size"), min_rel_diff=("max_rel_diff", "min"), median_rel_diff=("max_rel_diff", "median"),
                               fail_to_recover=("max_rel_diff", lambda x: int((x > 0.1).sum())))
print("\n", ms.round(3).to_string())

**Result: reproduced.** Each of 10 simple integrators ends 20% to 22% of its selectivity away from its
unperturbed output after silencing half of its units (persistent input), and each of 10 FORCE
networks trained to ramp after a transient input ends 13% to 76% away (median 41%); all exceed the
10% bound. The transient-input and cascaded integrators end about 50% away. The two models whose
parameters the text lacks were not built.

## fig6-modular-models-recover

> Modular networks of two coupled modules (attractor, integrator, trained recurrent network) recover
> after silencing of one module; the modular attractor drifts to a random fixed point when both
> modules are silenced.

Criterion: modular attractor and modular integrator with the Methods' parameters; 1 ms step with
Euler-Maruyama as the standard choice before the answer. Reproduced when, over 100 simulated trials per trial type, the modular attractor ends
at the instructed fixed point in more than 90% of trials after unilateral silencing and in 30% to
70% after bilateral silencing, and the modular integrator returns within 10% of its unperturbed
trajectory by 2 s after unilateral silencing. Answered 2026-10-21 as not given by the paper: the
noise, (n1) white noise of intensity 0.01, increment sd sqrt(0.01 dt)/tau, or (n2) one sample of
variance 0.01 per step, integrated like an input; and the step, (h1) 1 ms or (h2) 0.1 ms. The
outcome must hold under all four, else `methods_underspecified`. A trial ends at
the instructed fixed point when both modules have the instructed unit above the other one at 2 s.

In [ ]:
rows = []
STEPS = {"h1": 0.001, "h2": 0.0001}
for noise, nname in (("em", "n1"), ("sample", "n2")):
    for hname, dt in STEPS.items():
        for cond, sil in (("unperturbed", ()), ("unilateral", (0, 1)), ("bilateral", (0, 1, 2, 3))):
            ok = []
            for k, trial in enumerate(("right", "left")):
                _, R = L.modular_attractor(trial, silence=sil, seed=7 + k + 10 * len(sil), noise=noise, n_trials=100, dt=dt)
                end = R[:, -1]
                inst = (end[:, 0] > end[:, 1]) & (end[:, 2] > end[:, 3]) if trial == "right" else (end[:, 1] > end[:, 0]) & (end[:, 3] > end[:, 2])
                ok.append(inst)
            ok = np.concatenate(ok)
            rows.append({"noise": nname, "step": hname, "silencing": cond, "trials": len(ok), "instructed_fixed_point": ok.mean()})
att = pd.DataFrame(rows)
print(att.to_string(index=False))
mi = []
for hname, dt in STEPS.items():
    for trial in ("right", "left"):
        _, R0 = L.modular_integrator(trial, dt=dt); _, R1 = L.modular_integrator(trial, silence=(0, 1, 2, 3), dt=dt)
        sel0 = R0[-1, 0] - R0[-1, 1]; sel1 = R1[-1, 0] - R1[-1, 1]
        sel0b = R0[-1, 4] - R0[-1, 5]; sel1b = R1[-1, 4] - R1[-1, 5]
        mi.append({"step": hname, "trial": trial, "module1_unperturbed": sel0, "module1_silenced": sel1, "rel_diff_module1": abs(sel1 - sel0) / abs(sel0),
                   "module2_unperturbed": sel0b, "module2_after": sel1b, "rel_diff_module2": abs(sel1b - sel0b) / abs(sel0b)})
mi = pd.DataFrame(mi)
print("\nModular integrator, right minus left integrator unit at 2 s, module 1 silenced 0.2-1.0 s:")
print(mi.round(4).to_string(index=False))
judge6 = []
for nname in ("n1", "n2"):
    for hname in STEPS:
        a = att[(att.noise == nname) & (att.step == hname)].set_index("silencing").instructed_fixed_point
        m = mi[mi.step == hname].set_index("trial").rel_diff_module1 <= 0.1
        parts = {"attractor_unilateral": bool(a["unilateral"] > 0.9), "attractor_bilateral": bool(0.3 <= a["bilateral"] <= 0.7),
                 "integrator_right": bool(m["right"]), "integrator_left": bool(m["left"])}
        judge6.append({"noise": nname, "step": hname, **parts,
                       "outcome_here": "reproduced" if all(parts.values()) else "partially" if any(parts.values()) else "not"})
judge6 = pd.DataFrame(judge6)
print("\n", judge6.to_string(index=False))


**Result: partially reproduced.** The answer leaves the noise (n1, n2) and the step (h1, h2) open,
and the claims file requires one outcome over all four; all four give partial. Under n2 the
modular attractor ends at the instructed fixed point in 100% of trials after unilateral silencing
and in 48% (h1) or 50% (h2) after bilateral silencing, as the claim says. Under n1 the noise
dominates: it ends there in 41.5% to 51% of trials whatever the silencing (51% and 48.5%
unperturbed), so the unilateral part fails, and the bilateral fraction (43%, 44%) lies in the 30% to
70% band only because the network is at chance. The modular integrator with the printed matrix
restores lick right trials (0.2% from unperturbed at 2 s with 1 ms steps, 5.4% with 0.1 ms steps)
but not lick left ones (50% away under both steps): its recovery unit passes only right-preferring
activity.

## Blocked claims

- fig1d-late-delay-ipsilateral-bias, fig1d-earlier-photoinhibition-minimal,
  ed3-behaviour-recovery-interval, fig2b-unilateral-performance, fig2b-bilateral-performance
  (data_missing). They rest on the behavioural photoinhibition sessions of 5 and 13 mice
  (photoinhibition on 25% of trials, light ending at the go cue, 1.3 s light with a variable delay,
  four-spot unilateral and two-spot bilateral protocols). The dandiset holds only the recording
  sessions (40% light trials), which lack those conditions (data check). The behaviour during the
  recordings is shown above for reference only: in experiment 1 (4 mice) performance on early-delay
  and late-delay photoinhibition trials is 0.745 and 0.749 against 0.799 in control (paired p 0.21
  and 0.16), in line with fig1d-earlier-photoinhibition-minimal; in experiment 2 (7 mice) one spot
  with the hemispheres pooled gives 0.716 against 0.773 (paired p 0.059) and eight spots bilaterally
  0.580 (paired p 0.0001); in experiment 3 (3 mice) one spot pooled gives 0.705, four spots 0.756
  (left) and 0.830 (right), and two spots bilaterally 0.658 against 0.795 in control (paired p 0.045).
  These are other mice and sessions than the paper's behavioural cohort, with light on 40% of trials.
- ed5-photoactivation-recovery, ed5-photoactivation-rate-changes (data_missing). No
  Tlx_PL56-cre x Ai32 mouse is in the dandiset (genotypes in the data check).
- ed2d-spatial-extent (data_missing). The recordings of the 4 untrained mice are not in the
  dandiset.
- ed9f-reaction-time (data_missing). The analysis ran in its own section above: the lick trace of
  the experiment-2 sessions holds no licks, so the reaction times the claim needs are not there.
- fig5b-callosotomy-control-performance (data_missing). There are no sessions before the
  callosotomy, and no file marks the 7 mice as callosotomised.
- fig5b-callosotomy-unilateral-sensitivity, ed10-callosotomy-preparatory-activity,
  fig5-callosotomy-no-recovery, fig5f-callosotomy-cd-reduced (metadata_insufficient). The 33
  sessions of the 7 mice without a photostimulus record contain photoinhibition trials (silenced
  trials in the data check), but no file says which trials, where or when, so control and
  photoinhibition trials cannot be told apart. They hold 519 putative pyramidal units (paper 496,
  within tolerance, if these are the callosotomy mice, which no file says). Counted on all their
  trials, photoinhibition trials included, 331 to 340 are selective with correct trials (paper 254 on
  control trials) and 81 to 84 in the sample epoch (cell below); this is not the claim's measure.

In [ ]:
# ed10-callosotomy-preparatory-activity, the part that can be counted (verifier's suggestion; reference
# only). In the sessions without a photostimulus record control trials cannot be identified, so
# convention S is run on all their trials (lick-early dropped, K1, or kept, K2), with photoinhibition
# trials mixed in. Width classes are those of the data check.
cal_pyr = U[(U.exp == "no_photostim_record") & (U.cls == "pyr")]
rows = []
for early in ("drop", "keep"):
    for r in L.S_READINGS:
        st = L.selectivity_tests(cal_pyr, S, r["response"], r["outcomes"], early, r["equal_var"], conds=("unlabelled",))
        sig = st[["sample", "delay", "response"]] < 0.05
        rows.append({"early_lick": early, "reading": L.s_key(r), "units": len(st), "any": int(sig.any(axis=1).sum()),
                     "sample": int(sig["sample"].sum()), "delay": int(sig.delay.sum()), "response": int(sig.response.sum())})
print(pd.DataFrame(rows).to_string(index=False))
print("Putative pyramidal units:", len(cal_pyr), "(paper 496, tolerance 49.6); paper 254 selective (tolerance 25.4), on control trials.")

## Summary

| claim | outcome |
|---|---|
| unit-classification | partially reproduced |
| selective-fraction | not reproduced |
| fig1d-late-delay-ipsilateral-bias | blocked (data_missing) |
| fig1d-earlier-photoinhibition-minimal | blocked (data_missing) |
| ed3-behaviour-recovery-interval | blocked (data_missing) |
| ed4-return-to-trajectory | not reproduced |
| fig1f-spike-rate-recovery | reproduced |
| fig1g-selectivity-recovery | blocked (methods_underspecified) |
| ed5-photoactivation-recovery | blocked (data_missing) |
| ed5-photoactivation-rate-changes | blocked (data_missing) |
| ed2b-silencing-strength | not reproduced |
| ed2d-spatial-extent | blocked (data_missing) |
| ed2e-no-rebound | not reproduced |
| fig2b-unilateral-performance | blocked (data_missing) |
| fig2b-bilateral-performance | blocked (data_missing) |
| ed6a-bilateral-inconsistent-bias | reproduced |
| fig2-recordings-recovery | partially reproduced |
| fig2e-selectivity-recovery-times | not reproduced |
| ed7-larger-photoinhibition | blocked (methods_underspecified) |
| ed7e-roc-larger-photoinhibition | partially reproduced |
| cd-stable-over-epochs | blocked (methods_underspecified) |
| cd-captures-selectivity | blocked (methods_underspecified) |
| fig3c-cd-recovery-roc | not reproduced |
| ed8-roc-during-photoinhibition | partially reproduced |
| fig3c-cd-predicts-movement | reproduced |
| fig3d-persistent-mode | not reproduced |
| fig3e-ramping-mode | not reproduced |
| fig4-bilateral-cd-predicts-movement | reproduced |
| ed6b-session-bias-predicted | not reproduced |
| ed9f-reaction-time | blocked (data_missing) |
| ed9c-cd-window-robust | reproduced |
| fig5b-callosotomy-control-performance | blocked (data_missing) |
| fig5b-callosotomy-unilateral-sensitivity | blocked (metadata_insufficient) |
| ed10-callosotomy-preparatory-activity | blocked (metadata_insufficient) |
| fig5-callosotomy-no-recovery | blocked (metadata_insufficient) |
| fig5f-callosotomy-cd-reduced | blocked (metadata_insufficient) |
| ed1-monolithic-models-fail | reproduced |
| fig6-modular-models-recover | partially reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's. Conditions are read from the laser traces and
events, as the trial table has none. Session 258250_20141202 (interleaved, overlapping trial
sequences) and the three experiment-1 sessions without a photostimulus record are left out of
trial-based analyses. Details the claims file leaves open are run under each reading. The answers
of 2026-10-21 to the ten questions asked found every detail missing from the paper; the readings
they name are run, and a claim whose method-detail readings disagree is blocked. This changed two
outcomes: cd-captures-selectivity (not reproduced before) is blocked, as its v1 and v2 readings
differ, and ed9f-reaction-time, blocked before for want of a lick rule, was run over a range of
thresholds and is blocked because the lick trace holds no licks. Several
counts of included neurons and sessions differ from the paper's (55 and 143 neurons for the
recovery times against 138 to 173 and 221 to 293 here; 16 sessions with 6 to 20 neurons for the
activity space against 24 with 6 to 37), so the paper's selections are not those of the claims
file. The simulations use standard choices for the integration step, the FORCE target and feedback,
and the seeds; the modular attractor is run with both noise readings and both steps the answer
names. Photoinhibition-trial random splits use one seed per session.

Partial outcomes. Where a criterion defines no partial outcome (unit-classification,
selective-fraction, ed2b-silencing-strength, fig2-recordings-recovery, ed7e, ed8 and fig6), a claim
that is not reproduced is called partially reproduced when at least half of the parts of its
criterion hold under every reading that must agree (K1 and K2, and the other method details), and
not reproduced otherwise. This rule was written after the first verification round, where the
verifier judged several of these claims not reproduced; it moved selective-fraction (one count of
four under K1) and ed2b-silencing-strength (one part of four under K2) to not reproduced, and
leaves unit-classification (four of five), fig2-recordings-recovery (three of five),
ed7e-roc-larger-photoinhibition and ed8-roc-during-photoinhibition (three of four under K1, two
under K2) and fig6-modular-models-recover (two or three of four) partial.

After the first verification round. cd-stable-over-epochs is blocked: w from all correct control
trials gives 0.563 and 0.588, inside the band, and w from the training subset 0.534 to 0.554,
outside, and the claims file fixes neither. Under K2, 3% to 16% of the trials of each
photoinhibition condition of experiment 1, 6.5% to 7.5% in experiment 2 and 1% to 4% in experiment
3 have their light before its logged time, so K2 mixes trials without light
into the photoinhibition averages (data check); the claims file defines K2 with events at their
logged times, and a question asks whether that is meant.

What was found about the dandiset. It holds the recordings of the paper's 12 control mice (58
sessions, 890 putative pyramidal units as in the paper) and 33 sessions of 7 further mice that
match the callosotomy experiment, but not the behavioural photoinhibition sessions, the
photoactivation and untrained mice, or any session before callosotomy. About 33 units of
intermediate spike width are missing (15 against 50). The photostimulus is not in the trial table;
it is recorded as laser traces per site and events, from which the site, the number of spots and
the timing can be read, but in 36 sessions (the 33 of the callosotomy mice and 3 of experiment 1)
it is missing although those sessions contain silenced trials. No file says which mice were
callosotomised. In lick-early trials the photostimulus event is logged at the restarted delay; in
339 of 931 such trials it lies more than 0.1 s after the light, which came up to 10 s before the
logged go cue. One session has interleaved overlapping trial sequences; 28 units have
spike times out of time order; the files' `cell_type` labels intermediate-width units as `FS`; the
behaviour-only file is empty. No file holds lick times: the `lick_trace` of the 24 experiment-2
sessions used for the activity space stays within about 0.002 of its baseline when the mouse
licks, so licks cannot be read from it, while the experiment-3 and unlabelled sessions read as
references do carry licks.

In [ ]:
print(f"notebook ran in {time.time() - started:.0f} s")